In [ ]:

# =============================================================================
# LSTM V2 FORECASTING — CELL 1
# ENVIRONMENT AND FROZEN DATASET VERIFICATION
#
# NO MODEL TRAINING
# NO OFFICIAL VALIDATION OR TEST LOADING
# =============================================================================

from pathlib import Path
import hashlib
import json
import random
import platform

import numpy as np
import pandas as pd
import torch

from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

SEED = 20261022

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATA_DIR = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "release_candidate_corrected"
)

FREEZE_DIR = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "FINAL_RELEASE_FREEZE"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "LSTM_V2_Horizon"
)

TRAIN_FILE = DATA_DIR / "windows_W10_TRAIN.npz"

META_FILE = (
    DATA_DIR
    / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
)

CONTRACT_FILE = (
    DATA_DIR
    / "FEATURE_AND_DATASET_CONTRACT.json"
)

MANIFEST_FILE = (
    FREEZE_DIR
    / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)

LOCK_FILE = (
    FREEZE_DIR
    / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

HASH_FILE = DATA_DIR / "SHA256SUMS.json"


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()


def canonical_hash(value):
    payload = json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


for path in [
    TRAIN_FILE,
    META_FILE,
    CONTRACT_FILE,
    MANIFEST_FILE,
    LOCK_FILE,
    HASH_FILE
]:
    if not path.is_file():
        raise FileNotFoundError(path)

CONTRACT = read_json(CONTRACT_FILE)
MANIFEST = read_json(MANIFEST_FILE)
FREEZE_LOCK = read_json(LOCK_FILE)

assert MANIFEST["release_version"] == "2.0.0"
assert MANIFEST["release_status"] == (
    "FROZEN_CONTROLLED_SYNTHETIC_BENCHMARK"
)

assert (
    canonical_hash(MANIFEST)
    == FREEZE_LOCK["manifest_sha256_canonical"]
)

RECORDED_HASHES = read_json(HASH_FILE)

for filename, expected in RECORDED_HASHES.items():
    actual = sha256_file(DATA_DIR / filename)

    if actual != expected:
        raise RuntimeError(
            f"Dataset integrity failure: {filename}"
        )

for filename, expected in (
    MANIFEST["data_integrity"]["file_sha256"].items()
):
    actual = sha256_file(DATA_DIR / filename)

    if actual != expected:
        raise RuntimeError(
            f"Manifest integrity failure: {filename}"
        )

FEATURES = list(CONTRACT["features"])
FAMILIES = list(CONTRACT["attack_families"])

HORIZONS = [5, 10, 15]
HORIZON_TO_COLUMN = {5: 0, 10: 1, 15: 2}

assert len(FEATURES) == 14
assert len(FAMILIES) == 5

assert CONTRACT["history_minutes"] == 10
assert CONTRACT["forecast_horizons"] == HORIZONS

print("=" * 90)
print("LSTM V2 — FROZEN RELEASE VERIFIED")
print("=" * 90)
print("Release version:", MANIFEST["release_version"])
print("Torch version:", torch.__version__)
print("Python:", platform.python_version())
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("TRAIN source:", TRAIN_FILE)
print("Predictors:", len(FEATURES))
print("Horizons:", HORIZONS)
print("Dataset SHA256 verification: PASS")
print("Model training: NOT STARTED")
print("Official VALIDATION / TEST: NOT LOADED")


In [ ]:

# =============================================================================
# LSTM V2 — CELL 2
# LOAD AND VERIFY TRAIN-ONLY TEMPORAL SEQUENCES
#
# NO MODEL TRAINING
# =============================================================================

with np.load(TRAIN_FILE, allow_pickle=False) as pack:
    X_RAW = pack["X"].astype(np.float32)

    Y_HORIZONS = pack["y_horizons"].astype(np.float32)
    ONSET_CLASS = pack["onset_class"].astype(np.int16)
    FAMILY_CLASS = pack["family_class"].astype(np.int8)

    EPISODE_ID = (
        pd.Series(pack["episode_id"])
        .astype(str)
        .to_numpy()
    )

    MINUTE = pack["minute"].astype(np.int16)

META = pd.read_csv(META_FILE)
META["Episode_ID"] = META["Episode_ID"].astype(str)

TRAIN_META = (
    META.loc[META["Split"] == "TRAIN"]
    .set_index("Episode_ID")
)

assert TRAIN_META.index.is_unique

assert X_RAW.shape == (88186, 10, 14)
assert Y_HORIZONS.shape == (len(X_RAW), 3)

assert np.isfinite(X_RAW).all()
assert (X_RAW >= 0).all()

assert len(np.unique(EPISODE_ID)) == 1400
assert set(EPISODE_ID) == set(TRAIN_META.index)

assert np.array_equal(
    Y_HORIZONS[:, 0],
    ((ONSET_CLASS >= 1) & (ONSET_CLASS <= 5))
)

assert np.array_equal(
    Y_HORIZONS[:, 1],
    ((ONSET_CLASS >= 1) & (ONSET_CLASS <= 10))
)

assert np.array_equal(
    Y_HORIZONS[:, 2],
    ((ONSET_CLASS >= 1) & (ONSET_CLASS <= 15))
)

assert np.all(
    Y_HORIZONS[:, 0] <= Y_HORIZONS[:, 1]
)

assert np.all(
    Y_HORIZONS[:, 1] <= Y_HORIZONS[:, 2]
)

assert np.all(
    (ONSET_CLASS == 0) == (FAMILY_CLASS == 0)
)

SOURCE = (
    TRAIN_META.loc[EPISODE_ID, "Source"]
    .astype(str)
    .to_numpy()
)

CAL_GROUP = (
    TRAIN_META.loc[
        EPISODE_ID,
        "Calibration_Group_ID"
    ]
    .astype(str)
    .to_numpy()
)

CV_GROUPS = np.array([
    f"{src}::{grp}"
    for src, grp in zip(SOURCE, CAL_GROUP)
])

assert len(np.unique(CV_GROUPS)) == 35

# The episode ID and minute are audit identifiers only.
# Neither is supplied to the neural network.

print("=" * 90)
print("LSTM V2 — TRAIN ARRAYS VERIFIED")
print("=" * 90)

print("Sequence shape:", X_RAW.shape)
print("Targets:", Y_HORIZONS.shape)
print("Unique episodes:", len(np.unique(EPISODE_ID)))
print("Calibration groups:", len(np.unique(CV_GROUPS)))

for horizon, col in HORIZON_TO_COLUMN.items():
    print(
        f"H{horizon} positive prevalence:",
        f"{Y_HORIZONS[:, col].mean():.6f}"
    )

print("\nPredictor example:")
print("Oldest observed minute:", X_RAW[0, 0, :3])
print("Newest observed minute:", X_RAW[0, -1, :3])

print("\nModel training: NOT STARTED")


In [ ]:

# =============================================================================
# LSTM V2 — CELL 3
# FEATURE-WISE PREPROCESSING AND ROBUST SCALER
#
# No future observations.
# No validation/test fitting.
# No model training.
# =============================================================================

class TemporalRobustScaler:

    def __init__(self, clip_value=8.0):
        self.clip_value = float(clip_value)
        self.median_ = None
        self.scale_ = None

    def fit(self, X):
        assert X.ndim == 3
        assert X.shape[-1] == 14

        # All values are nonnegative from the frozen contract.
        z = np.log1p(X.astype(np.float32))

        # Feature-wise statistics across windows and timesteps.
        # Safe for the supplied training subset, but ONLY
        # use the fold-fitting partition during CV.
        flat = z.reshape(-1, z.shape[-1])

        q25, q50, q75 = np.quantile(
            flat,
            [0.25, 0.50, 0.75],
            axis=0
        )

        self.median_ = q50.astype(np.float32)

        self.scale_ = np.maximum(
            (q75 - q25).astype(np.float32),
            0.05
        )

        return self

    def transform(self, X):
        if self.median_ is None:
            raise RuntimeError("Scaler has not been fitted.")

        z = np.log1p(X.astype(np.float32))

        z = (
            z - self.median_[None, None, :]
        ) / self.scale_[None, None, :]

        z = np.clip(
            z,
            -self.clip_value,
            self.clip_value
        )

        return np.ascontiguousarray(
            z,
            dtype=np.float32
        )

    def state_dict(self):
        if self.median_ is None:
            raise RuntimeError("Scaler has not been fitted.")

        return {
            "median": self.median_.tolist(),
            "scale": self.scale_.tolist(),
            "clip_value": self.clip_value,
            "feature_names": FEATURES
        }


# Preprocessing smoke test on TRAIN data.
# This full-TRAIN scaler is for future final training only.
# Cross-validation must fit a new scaler per fitting fold.

FINAL_TRAIN_SCALER = TemporalRobustScaler(
    clip_value=8.0
).fit(X_RAW)

X_SCALED_PREVIEW = FINAL_TRAIN_SCALER.transform(
    X_RAW[:128]
)

assert X_SCALED_PREVIEW.shape == (128, 10, 14)
assert np.isfinite(X_SCALED_PREVIEW).all()

# Verify a single observation receives identical
# transformation regardless of its timestep position.
same_value = np.repeat(
    X_RAW[:1, :1, :],
    10,
    axis=1
)

same_value_scaled = FINAL_TRAIN_SCALER.transform(
    same_value
)

assert np.allclose(
    same_value_scaled[:, 0, :],
    same_value_scaled[:, 9, :]
)

print("=" * 90)
print("LSTM V2 — PREPROCESSING VERIFIED")
print("=" * 90)

print("Log transform: log1p")
print("Scaling: per-feature median / IQR")
print("Clip range: [-8, +8]")
print("Full-TRAIN preview shape:", X_SCALED_PREVIEW.shape)

print("\nMedian parameters:", FINAL_TRAIN_SCALER.median_)
print("IQR scales:", FINAL_TRAIN_SCALER.scale_)

print("\nFold-level scalers: WILL BE FIT PER FOLD")
print("No VALIDATION or TEST preprocessing performed")
print("Model training: NOT STARTED")


In [ ]:

# =============================================================================
# LSTM V2 — CELL 4
# TRAIN-ONLY CALIBRATION-GROUP-DISJOINT CV
#
# Identical grouping principle to XGBoost V2.
# No training.
# =============================================================================

from sklearn.model_selection import GroupKFold

N_SPLITS = 5

CV_FOLDS = list(
    GroupKFold(n_splits=N_SPLITS).split(
        X_RAW,
        Y_HORIZONS[:, 1],
        groups=CV_GROUPS
    )
)

FOLD_ASSIGNMENTS = np.full(
    len(X_RAW),
    -1,
    dtype=np.int8
)

EPISODE_FAMILY = (
    TRAIN_META.loc[EPISODE_ID, "Family"]
    .astype(str)
    .to_numpy()
)

ALL_SOURCES = set(np.unique(SOURCE))

cv_rows = []

for fold_number, (fit_idx, hold_idx) in enumerate(
    CV_FOLDS,
    start=1
):
    fit_groups = set(CV_GROUPS[fit_idx])
    hold_groups = set(CV_GROUPS[hold_idx])

    fit_episodes = set(EPISODE_ID[fit_idx])
    hold_episodes = set(EPISODE_ID[hold_idx])

    assert fit_groups.isdisjoint(hold_groups)
    assert fit_episodes.isdisjoint(hold_episodes)

    assert set(SOURCE[fit_idx]) == ALL_SOURCES
    assert set(SOURCE[hold_idx]) == ALL_SOURCES

    FOLD_ASSIGNMENTS[hold_idx] = fold_number

    row = {
        "Fold": fold_number,
        "Fit_Windows": len(fit_idx),
        "Holdout_Windows": len(hold_idx),
        "Fit_Groups": len(fit_groups),
        "Holdout_Groups": len(hold_groups),
        "Fit_Episodes": len(fit_episodes),
        "Holdout_Episodes": len(hold_episodes)
    }

    for h, col in HORIZON_TO_COLUMN.items():
        fit_labels = Y_HORIZONS[fit_idx, col]
        hold_labels = Y_HORIZONS[hold_idx, col]

        assert len(np.unique(fit_labels)) == 2
        assert len(np.unique(hold_labels)) == 2

        row[f"H{h}_Holdout_Prevalence"] = float(
            hold_labels.mean()
        )

    for family in FAMILIES:
        row[f"Episodes_{family}"] = len(
            set(
                EPISODE_ID[hold_idx][
                    EPISODE_FAMILY[hold_idx] == family
                ]
            )
        )

        assert row[f"Episodes_{family}"] > 0

    for source in sorted(ALL_SOURCES):
        row[f"Windows_{source}"] = int(
            np.sum(SOURCE[hold_idx] == source)
        )

    cv_rows.append(row)

assert np.all(FOLD_ASSIGNMENTS > 0)

fold_group_counts = (
    pd.DataFrame({
        "Calibration_Group": CV_GROUPS,
        "Fold": FOLD_ASSIGNMENTS
    })
    .groupby("Calibration_Group")["Fold"]
    .nunique()
)

fold_episode_counts = (
    pd.DataFrame({
        "Episode_ID": EPISODE_ID,
        "Fold": FOLD_ASSIGNMENTS
    })
    .groupby("Episode_ID")["Fold"]
    .nunique()
)

assert (fold_group_counts == 1).all()
assert (fold_episode_counts == 1).all()

CV_SUMMARY = pd.DataFrame(cv_rows)

print("=" * 90)
print("LSTM V2 — CROSS-VALIDATION VERIFIED")
print("=" * 90)

print(
    CV_SUMMARY[
        [
            "Fold",
            "Fit_Windows",
            "Holdout_Windows",
            "Fit_Groups",
            "Holdout_Groups",
            "H5_Holdout_Prevalence",
            "H10_Holdout_Prevalence",
            "H15_Holdout_Prevalence"
        ]
    ].round(5).to_string(index=False)
)

print("\nAttack-family distribution by holdout fold:")

print(
    CV_SUMMARY[
        ["Fold"] +
        [f"Episodes_{family}" for family in FAMILIES]
    ].to_string(index=False)
)

print("\nSource distribution by holdout fold:")

print(
    CV_SUMMARY[
        ["Fold"] +
        [f"Windows_{src}" for src in sorted(ALL_SOURCES)]
    ].to_string(index=False)
)

print("\nCalibration-group overlap: ZERO")
print("Episode overlap: ZERO")
print("Official VALIDATION / TEST: NOT LOADED")
print("Training: NOT STARTED")


In [ ]:

# =============================================================================
# LSTM V2 — CELL 5
# DEFINE MONOTONIC THREE-HORIZON LSTM
#
# INPUT:
#   (batch, 10, 14)
#
# OUTPUT:
#   Three cumulative probabilities:
#       P(attack within 5 minutes)
#       P(attack within 10 minutes)
#       P(attack within 15 minutes)
#
# MATHEMATICAL CONSTRAINT:
#   0 <= P(H5) <= P(H10) <= P(H15) <= 1
#
# NO TRAINING
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class MonotonicHorizonHead(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            3
        )

    def forward(self, hidden):

        # Conditional event probabilities over intervals.
        conditional_logits = self.linear(hidden)

        q = torch.sigmoid(conditional_logits)

        # Numerically stable cumulative survival calculation.
        survival = torch.cumprod(
            1.0 - q,
            dim=-1
        )

        cumulative = 1.0 - survival

        return cumulative, conditional_logits


class LSTMHorizonForecaster(nn.Module):

    def __init__(
        self,
        input_dim=14,
        hidden_dim=64,
        num_layers=2,
        dropout=0.2,
        dense_dim=32
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, dense_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.head = MonotonicHorizonHead(
            input_dim=dense_dim
        )

    def forward(self, x):

        if x.ndim != 3:
            raise ValueError(
                f"Expected 3D input, got {x.shape}"
            )

        if x.shape[1:] != (10, 14):
            raise ValueError(
                f"Expected (batch,10,14), got {x.shape}"
            )

        output, (h_n, c_n) = self.lstm(x)

        # Last-layer final hidden state, corresponding
        # to the newest observed minute of each window.
        final_hidden = h_n[-1]

        representation = self.projection(
            final_hidden
        )

        probabilities, conditional_logits = self.head(
            representation
        )

        return {
            "probabilities": probabilities,
            "conditional_logits": conditional_logits
        }


MODEL_CONFIG = {
    "input_dim": 14,
    "hidden_dim": 64,
    "num_layers": 2,
    "dropout": 0.2,
    "dense_dim": 32
}

model = LSTMHorizonForecaster(
    **MODEL_CONFIG
).to(DEVICE)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

print("=" * 90)
print("LSTM V2 — ARCHITECTURE DEFINED")
print("=" * 90)

print(model)

print("\nTrainable parameters:", parameter_count)
print("Input: (batch, 10, 14)")
print("Output: (batch, 3)")
print("Horizon probability constraint: MONOTONIC")
print("Model weights: RANDOMLY INITIALIZED")
print("Training: NOT STARTED")


In [ ]:

# =============================================================================
# LSTM V2 — CELL 6
# FORWARD PASS + LOSS + PROBABILITY CONSISTENCY SMOKE TEST
#
# NO OPTIMIZER
# NO BACKWARD PASS
# NO WEIGHT UPDATES
# NO MODEL TRAINING
# =============================================================================

def horizon_labels_to_interval_class(y):
    """
    Convert cumulative horizon labels into one of 4 classes.

    0: Onset within 1-5 minutes
    1: Onset within 6-10 minutes
    2: Onset within 11-15 minutes
    3: No onset within 15 minutes
    """

    if y.ndim != 2 or y.shape[1] != 3:
        raise ValueError("Expected labels shape (batch,3).")

    if not torch.all((y == 0) | (y == 1)):
        raise ValueError("Labels must be binary.")

    if not torch.all(y[:, 0] <= y[:, 1]):
        raise ValueError("Horizon labels not nested.")

    if not torch.all(y[:, 1] <= y[:, 2]):
        raise ValueError("Horizon labels not nested.")

    classes = torch.full(
        (len(y),),
        3,
        device=y.device,
        dtype=torch.long
    )

    classes[y[:, 2] == 1] = 2
    classes[y[:, 1] == 1] = 1
    classes[y[:, 0] == 1] = 0

    return classes


def interval_nll_from_logits(
    conditional_logits,
    target_classes
):
    """
    Stable discrete-time survival likelihood.

    q0 = event in (0,5]
    q1 = event in (5,10] conditional on survival
    q2 = event in (10,15] conditional on survival

    Target:
      class 0 -> log q0
      class 1 -> log(1-q0) + log q1
      class 2 -> log(1-q0) + log(1-q1) + log q2
      class 3 -> log(1-q0) + log(1-q1) + log(1-q2)
    """

    z = conditional_logits

    log_event = F.logsigmoid(z)
    log_survive = F.logsigmoid(-z)

    log_probs = torch.stack(
        [
            log_event[:, 0],

            log_survive[:, 0]
            + log_event[:, 1],

            log_survive[:, 0]
            + log_survive[:, 1]
            + log_event[:, 2],

            log_survive.sum(dim=1)
        ],
        dim=1
    )

    selected_log_probs = log_probs.gather(
        1,
        target_classes.unsqueeze(1)
    ).squeeze(1)

    return -selected_log_probs.mean()


# =============================================================================
# A — CHECK TARGET ENCODING WITH KNOWN EXAMPLES
# =============================================================================

example_labels = torch.tensor(
    [
        [1, 1, 1],
        [0, 1, 1],
        [0, 0, 1],
        [0, 0, 0]
    ],
    dtype=torch.float32
)

expected_classes = torch.tensor(
    [0, 1, 2, 3],
    dtype=torch.long
)

actual_classes = horizon_labels_to_interval_class(
    example_labels
)

assert torch.equal(
    actual_classes,
    expected_classes
)

# =============================================================================
# B — VERIFY MATHEMATICAL PROBABILITY NORMALIZATION
# =============================================================================

model.eval()

with torch.no_grad():

    dummy_x = torch.randn(
        16,
        10,
        14,
        device=DEVICE
    )

    dummy_output = model(dummy_x)

    p = dummy_output["probabilities"]
    logits = dummy_output["conditional_logits"]

    assert p.shape == (16, 3)
    assert logits.shape == (16, 3)

    assert torch.isfinite(p).all()
    assert torch.isfinite(logits).all()

    assert (p >= 0).all()
    assert (p <= 1).all()

    assert torch.all(
        p[:, 0] <= p[:, 1] + 1e-7
    )

    assert torch.all(
        p[:, 1] <= p[:, 2] + 1e-7
    )

    interval_probs = torch.stack(
        [
            p[:, 0],
            p[:, 1] - p[:, 0],
            p[:, 2] - p[:, 1],
            1.0 - p[:, 2]
        ],
        dim=1
    )

    assert torch.all(
        interval_probs >= -1e-7
    )

    assert torch.allclose(
        interval_probs.sum(dim=1),
        torch.ones(16, device=DEVICE),
        atol=1e-6
    )

# =============================================================================
# C — CHECK A REAL TRAIN BATCH WITHOUT MODEL FITTING
# =============================================================================

preview_size = 32

preview_raw = X_RAW[:preview_size]

preview_scaled = FINAL_TRAIN_SCALER.transform(
    preview_raw
)

preview_x = torch.from_numpy(
    preview_scaled
).to(DEVICE)

preview_y = torch.from_numpy(
    Y_HORIZONS[:preview_size]
).to(DEVICE)

preview_classes = horizon_labels_to_interval_class(
    preview_y
)

with torch.no_grad():

    preview_out = model(preview_x)

    preview_loss = interval_nll_from_logits(
        preview_out["conditional_logits"],
        preview_classes
    )

    assert torch.isfinite(preview_loss)

    # Independent cross-check:
    # NLL should equal the log probability
    # of the corresponding 4-class interval.
    prediction_p = preview_out["probabilities"]

    interval_probability = torch.stack(
        [
            prediction_p[:, 0],
            prediction_p[:, 1] - prediction_p[:, 0],
            prediction_p[:, 2] - prediction_p[:, 1],
            1.0 - prediction_p[:, 2]
        ],
        dim=1
    )

    gathered = interval_probability.gather(
        1,
        preview_classes.unsqueeze(1)
    ).squeeze(1)

    independently_calculated_loss = (
        -torch.log(
            gathered.clamp_min(1e-12)
        ).mean()
    )

    assert torch.allclose(
        preview_loss,
        independently_calculated_loss,
        atol=1e-5,
        rtol=1e-5
    )

print("=" * 90)
print("LSTM V2 — SMOKE TEST COMPLETE")
print("=" * 90)

print("Untrained forward pass: PASS")
print("Input shape:", tuple(preview_x.shape))
print(
    "Probability output shape:",
    tuple(preview_out["probabilities"].shape)
)

print(
    "Interval-class encoding:",
    actual_classes.tolist()
)

print(
    "Untrained NLL:",
    round(float(preview_loss.item()), 6)
)

print("Interval probabilities sum to one: PASS")
print("P(H5) <= P(H10) <= P(H15): PASS")
print("Loss cross-check: PASS")
print("NaN / infinity check: PASS")

print("\nModel training: NOT STARTED")
print("Backward pass: NOT PERFORMED")
print("Optimizer steps: ZERO")
print("Official VALIDATION / TEST: NOT LOADED")
print("Frozen dataset: UNCHANGED")


In [ ]:

# =============================================================================
# LSTM V2 — CELL 7
# CONTROLLED TRAINING SANITY CHECK
#
# WARNING: THIS CELL TRAINS A DIAGNOSTIC LSTM WHEN RUN.
#
# Uses:
#   TRAIN partition only
#   Fold 1 from the existing group-disjoint CV
#   A scaler fitted only on fold 1 fitting groups
#   The previously verified monotonic LSTM architecture and NLL
#
# Limits:
#   8 epochs maximum
#   No Optuna
#   No official VALIDATION
#   No TEST
#   No final model checkpoint
# =============================================================================

import copy
import gc
import random
import time

import numpy as np
import pandas as pd

import torch
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

# =============================================================================
# 7A — VERIFY PREREQUISITES
# =============================================================================

NEEDED = [
    "X_RAW",
    "Y_HORIZONS",
    "CV_FOLDS",
    "CV_GROUPS",
    "EPISODE_ID",
    "SOURCE",
    "TemporalRobustScaler",
    "LSTMHorizonForecaster",
    "horizon_labels_to_interval_class",
    "interval_nll_from_logits",
    "MODEL_CONFIG",
    "DEVICE"
]

missing = [
    key for key in NEEDED
    if key not in globals()
]

if missing:
    raise RuntimeError(
        "Run Cells 1–6 first. Missing: "
        + str(missing)
    )

assert X_RAW.shape == (88186, 10, 14)
assert Y_HORIZONS.shape == (88186, 3)
assert len(CV_FOLDS) == 5
assert np.isfinite(X_RAW).all()

# =============================================================================
# 7B — FIXED SANITY-CHECK SETTINGS
# =============================================================================

SANITY_SEED = 20261022

SANITY_FOLD = 1

MAX_EPOCHS = 8
BATCH_SIZE = 256

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3

GRAD_CLIP_NORM = 1.0

NUM_WORKERS = 0

random.seed(SANITY_SEED)
np.random.seed(SANITY_SEED)
torch.manual_seed(SANITY_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SANITY_SEED)

TRAIN_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 95)
print("LSTM V2 — CONTROLLED TRAINING SANITY CHECK")
print("=" * 95)

print("Torch:", torch.__version__)
print("Device:", TRAIN_DEVICE)
print("Fold:", SANITY_FOLD)
print("Maximum epochs:", MAX_EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)

if TRAIN_DEVICE.type == "cpu":
    print(
        "\nNOTE: CPU runtime detected. "
        "This short check can still run on CPU."
    )

# =============================================================================
# 7C — SELECT ONE GROUP-DISJOINT TRAIN FOLD
# =============================================================================

fit_idx, hold_idx = CV_FOLDS[SANITY_FOLD - 1]

fit_groups = set(CV_GROUPS[fit_idx])
hold_groups = set(CV_GROUPS[hold_idx])

fit_episodes = set(EPISODE_ID[fit_idx])
hold_episodes = set(EPISODE_ID[hold_idx])

assert fit_groups.isdisjoint(hold_groups)
assert fit_episodes.isdisjoint(hold_episodes)

assert set(SOURCE[fit_idx]) == set(SOURCE[hold_idx])

print("\nFold isolation:")
print("Fit windows:", len(fit_idx))
print("Holdout windows:", len(hold_idx))
print("Fit groups:", len(fit_groups))
print("Holdout groups:", len(hold_groups))
print("Shared groups:", len(fit_groups & hold_groups))
print("Shared episodes:", len(fit_episodes & hold_episodes))

# =============================================================================
# 7D — FIT SCALER ON FOLD-FITTING DATA ONLY
# =============================================================================

SANITY_SCALER = TemporalRobustScaler(
    clip_value=8.0
)

SANITY_SCALER.fit(
    X_RAW[fit_idx]
)

X_FIT = SANITY_SCALER.transform(
    X_RAW[fit_idx]
)

X_HOLD = SANITY_SCALER.transform(
    X_RAW[hold_idx]
)

Y_FIT = Y_HORIZONS[fit_idx].astype(np.float32)
Y_HOLD = Y_HORIZONS[hold_idx].astype(np.float32)

assert X_FIT.shape[1:] == (10, 14)
assert X_HOLD.shape[1:] == (10, 14)

assert np.isfinite(X_FIT).all()
assert np.isfinite(X_HOLD).all()

# Labels are nested cumulative outcomes.
assert np.all(Y_FIT[:, 0] <= Y_FIT[:, 1])
assert np.all(Y_FIT[:, 1] <= Y_FIT[:, 2])

assert np.all(Y_HOLD[:, 0] <= Y_HOLD[:, 1])
assert np.all(Y_HOLD[:, 1] <= Y_HOLD[:, 2])

print("\nPreprocessing:")
print("Scaler fitted on: FOLD FIT ONLY")
print("Fit shape:", X_FIT.shape)
print("Holdout shape:", X_HOLD.shape)

# =============================================================================
# 7E — CREATE DATALOADERS
# =============================================================================

fit_dataset = TensorDataset(
    torch.from_numpy(X_FIT),
    torch.from_numpy(Y_FIT)
)

hold_dataset = TensorDataset(
    torch.from_numpy(X_HOLD),
    torch.from_numpy(Y_HOLD)
)

fit_generator = torch.Generator()
fit_generator.manual_seed(SANITY_SEED)

fit_loader = DataLoader(
    fit_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=fit_generator,
    num_workers=NUM_WORKERS,
    pin_memory=(TRAIN_DEVICE.type == "cuda")
)

hold_loader = DataLoader(
    hold_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(TRAIN_DEVICE.type == "cuda")
)

# =============================================================================
# 7F — CREATE FRESH DIAGNOSTIC MODEL
#
# The untrained model object from Cell 5 is NOT reused.
# This avoids accidentally carrying weights into later runs.
# =============================================================================

SANITY_MODEL = LSTMHorizonForecaster(
    **MODEL_CONFIG
).to(TRAIN_DEVICE)

optimizer = torch.optim.AdamW(
    SANITY_MODEL.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# No class weighting or oversampling in this initial check.

print("\nDiagnostic LSTM parameters:", sum(
    p.numel()
    for p in SANITY_MODEL.parameters()
))

# =============================================================================
# 7G — TRAIN ONE EPOCH
# =============================================================================

def train_sanity_epoch(model, loader, optimizer, device):

    model.train()

    total_loss = 0.0
    total_examples = 0

    max_gradient_norm = 0.0

    for xb, yb in loader:

        xb = xb.to(
            device,
            non_blocking=True
        )

        yb = yb.to(
            device,
            non_blocking=True
        )

        classes = horizon_labels_to_interval_class(
            yb
        )

        optimizer.zero_grad(set_to_none=True)

        result = model(xb)

        loss = interval_nll_from_logits(
            result["conditional_logits"],
            classes
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Nonfinite training loss detected."
            )

        loss.backward()

        for name, parameter in model.named_parameters():

            if (
                parameter.grad is not None
                and not torch.isfinite(parameter.grad).all()
            ):
                raise RuntimeError(
                    f"Nonfinite gradient: {name}"
                )

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_NORM,
            error_if_nonfinite=True
        )

        max_gradient_norm = max(
            max_gradient_norm,
            float(grad_norm.item())
        )

        optimizer.step()

        batch_size = len(xb)

        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size

    return {
        "loss": total_loss / total_examples,
        "max_preclip_gradient_norm": max_gradient_norm
    }


# =============================================================================
# 7H — EVALUATE FOLD HOLDOUT
#
# This is internal TRAIN CV monitoring only.
# No early stopping or hyperparameter selection in Cell 7.
# =============================================================================

@torch.inference_mode()
def evaluate_sanity(model, loader, device):

    model.eval()

    total_loss = 0.0
    total_examples = 0

    probabilities = []
    labels = []

    for xb, yb in loader:

        xb = xb.to(
            device,
            non_blocking=True
        )

        yb_device = yb.to(
            device,
            non_blocking=True
        )

        classes = horizon_labels_to_interval_class(
            yb_device
        )

        result = model(xb)

        loss = interval_nll_from_logits(
            result["conditional_logits"],
            classes
        )

        p = result["probabilities"]

        if not torch.isfinite(p).all():
            raise RuntimeError(
                "Nonfinite predicted probabilities."
            )

        if not torch.all(
            p[:, 0] <= p[:, 1] + 1e-6
        ):
            raise RuntimeError(
                "H5/H10 probability ordering violated."
            )

        if not torch.all(
            p[:, 1] <= p[:, 2] + 1e-6
        ):
            raise RuntimeError(
                "H10/H15 probability ordering violated."
            )

        probabilities.append(
            p.detach().cpu().numpy()
        )

        labels.append(
            yb.numpy()
        )

        batch_size = len(xb)

        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size

    y_true = np.concatenate(
        labels,
        axis=0
    )

    y_prob = np.concatenate(
        probabilities,
        axis=0
    )

    ap = [
        float(
            average_precision_score(
                y_true[:, i],
                y_prob[:, i]
            )
        )
        for i in range(3)
    ]

    auc = [
        float(
            roc_auc_score(
                y_true[:, i],
                y_prob[:, i]
            )
        )
        for i in range(3)
    ]

    return {
        "loss": total_loss / total_examples,
        "AP_H5": ap[0],
        "AP_H10": ap[1],
        "AP_H15": ap[2],
        "AUC_H5": auc[0],
        "AUC_H10": auc[1],
        "AUC_H15": auc[2],
        "monotonic": True
    }


# =============================================================================
# 7I — RUN FIXED 8-EPOCH SANITY TRAINING
#
# No checkpoint selection, no early stopping, no model release.
# =============================================================================

SANITY_HISTORY = []

start_time = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):

    train_result = train_sanity_epoch(
        SANITY_MODEL,
        fit_loader,
        optimizer,
        TRAIN_DEVICE
    )

    hold_result = evaluate_sanity(
        SANITY_MODEL,
        hold_loader,
        TRAIN_DEVICE
    )

    row = {
        "Epoch": epoch,
        "Train_NLL": train_result["loss"],
        "Holdout_NLL": hold_result["loss"],
        "H5_AP": hold_result["AP_H5"],
        "H10_AP": hold_result["AP_H10"],
        "H15_AP": hold_result["AP_H15"],
        "H5_AUC": hold_result["AUC_H5"],
        "H10_AUC": hold_result["AUC_H10"],
        "H15_AUC": hold_result["AUC_H15"],
        "Max_Preclip_Gradient_Norm": train_result[
            "max_preclip_gradient_norm"
        ]
    }

    SANITY_HISTORY.append(row)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train NLL={row['Train_NLL']:.5f} | "
        f"Hold NLL={row['Holdout_NLL']:.5f} | "
        f"H5 AP={row['H5_AP']:.5f} | "
        f"H10 AP={row['H10_AP']:.5f} | "
        f"H15 AP={row['H15_AP']:.5f}"
    )

elapsed = time.perf_counter() - start_time

SANITY_HISTORY_DF = pd.DataFrame(
    SANITY_HISTORY
)

# =============================================================================
# 7J — FINAL SANITY ASSESSMENT
#
# Do not automatically declare predictive success based only
# on a decreasing training loss.
# =============================================================================

loss_finite = bool(
    np.isfinite(
        SANITY_HISTORY_DF[
            ["Train_NLL", "Holdout_NLL"]
        ].to_numpy()
    ).all()
)

ap_finite = bool(
    np.isfinite(
        SANITY_HISTORY_DF[
            ["H5_AP", "H10_AP", "H15_AP"]
        ].to_numpy()
    ).all()
)

initial_train_loss = float(
    SANITY_HISTORY_DF.iloc[0]["Train_NLL"]
)

final_train_loss = float(
    SANITY_HISTORY_DF.iloc[-1]["Train_NLL"]
)

train_loss_decreased = (
    final_train_loss < initial_train_loss
)

print("\n" + "=" * 95)
print("LSTM V2 — SANITY TRAINING COMPLETE")
print("=" * 95)

print("Device:", TRAIN_DEVICE)
print("Completed epochs:", len(SANITY_HISTORY_DF))
print("Elapsed seconds:", round(elapsed, 2))

print("\nInitial TRAIN NLL:", round(initial_train_loss, 6))
print("Final TRAIN NLL:", round(final_train_loss, 6))
print("TRAIN loss decreased:", train_loss_decreased)

print("\nLast epoch:")
print(
    SANITY_HISTORY_DF.tail(1).round(6).to_string(
        index=False
    )
)

print("\nBest monitored fold-holdout AP values:")
print(
    SANITY_HISTORY_DF[
        ["H5_AP", "H10_AP", "H15_AP"]
    ].max().round(6).to_dict()
)

print("\nNumerical checks:")
print("Finite losses:", loss_finite)
print("Finite AP metrics:", ap_finite)
print("Monotonic horizon outputs: PASS")

if not loss_finite or not ap_finite:
    raise RuntimeError(
        "LSTM sanity check failed numerical verification."
    )

if not train_loss_decreased:
    print(
        "\nWARNING: Training loss did not decrease "
        "across the 8 epochs. Investigate before CV."
    )

print("\nDiagnostic model only: NOT A FINAL MODEL")
print("No hyperparameter optimization performed")
print("Official VALIDATION: NOT ACCESSED")
print("Official TEST: NOT ACCESSED")
print("Frozen dataset: UNCHANGED")

print("\nNEXT: Review the loss and AP curves before")
print("starting five-fold LSTM model selection.")


In [ ]:

import json
from pathlib import Path

saved_path = Path(
    "/content/drive/MyDrive/network-attack-forecasting/"
    "Synthetic_NAF_v2/models/LSTM_V2_Horizon/"
    "train_cv_selection/SEARCH_PROTOCOL.json"
)

if not saved_path.exists():
    print("No saved protocol found.")
else:
    saved = json.loads(saved_path.read_text())

    print("SAVED ENVIRONMENT")
    print("Device:", saved.get("device"))
    print("PyTorch:", saved.get("torch_version"))

    print("\nCURRENT ENVIRONMENT")
    print("Device:", str(DEVICE_CV))
    print("PyTorch:", torch.__version__)

    if "PROTOCOL" in globals():
        print("\nDIFFERENT SETTINGS:")

        differences = []

        for key in sorted(set(saved) | set(PROTOCOL)):
            if saved.get(key) != PROTOCOL.get(key):
                differences.append(key)
                print(f"\n{key}")
                print("Saved:", str(saved.get(key))[:350])
                print("Current:", str(PROTOCOL.get(key))[:350])

        if not differences:
            print("No differences found.")

        print("\nTotal differing fields:", len(differences))

    completed = sorted(
        saved_path.parent.glob("C*/FOLD_*_RESULT.json")
    )

    print("\nSaved fold-result files:", len(completed))

    for path in completed:
        try:
            result = json.loads(path.read_text())
            print(
                path.parent.name,
                path.stem,
                "Best epoch:",
                result.get("best_epoch"),
                "H5 AP:",
                round(result["selected_metrics"]["H5_AP"], 5),
                "H10 AP:",
                round(result["selected_metrics"]["H10_AP"], 5)
            )
        except Exception as exc:
            print("Unreadable result:", path, str(exc))


In [ ]:

# ==============================================================================================
# LSTM V2 — CELL 8
# FIVE-FOLD TRAIN-ONLY HYPERPARAMETER SELECTION
#
# WARNING: EXECUTING THIS CELL TRAINS NEURAL NETWORKS.
#
# Protocol:
#   8 predefined configurations
#   5 calibration-group-disjoint TRAIN folds
#   Maximum 35 epochs per fold
#   Early stopping: fold-holdout NLL, patience 7
#   Configuration selection: mean of H5 and H10 AP
#   Final epoch count: median of selected fold epochs
#
# No official VALIDATION or TEST access.
# No dataset regeneration.
# No final full-TRAIN model fitting.
#
# Completed folds are saved and skipped on reruns.
# An interrupted fold is restarted from its beginning.
# ==============================================================================================

import os
import gc
import json
import copy
import time
import random
import hashlib
import platform

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import torch
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)

# ==============================================================================================
# 8A — VERIFY NOTEBOOK STATE
# ==============================================================================================

REQUIRED_OBJECTS = [
    "X_RAW",
    "Y_HORIZONS",
    "EPISODE_ID",
    "CV_GROUPS",
    "CV_FOLDS",
    "FEATURES",
    "TemporalRobustScaler",
    "LSTMHorizonForecaster",
    "horizon_labels_to_interval_class",
    "interval_nll_from_logits",
    "PROJECT_ROOT",
    "DATA_DIR",
    "FREEZE_DIR",
]

missing = [
    name for name in REQUIRED_OBJECTS
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run LSTM Cells 1–7 first. Missing: "
        + ", ".join(missing)
    )

assert X_RAW.shape == (88186, 10, 14)
assert Y_HORIZONS.shape == (88186, 3)
assert len(CV_FOLDS) == 5

assert np.isfinite(X_RAW).all()
assert np.all(X_RAW >= 0)

assert np.all(Y_HORIZONS[:, 0] <= Y_HORIZONS[:, 1])
assert np.all(Y_HORIZONS[:, 1] <= Y_HORIZONS[:, 2])

# ==============================================================================================
# 8B — FIXED SEARCH PROTOCOL
# ==============================================================================================

SEARCH_SEED = 20261022

MAX_EPOCHS = 35
PATIENCE = 7
MIN_DELTA = 1e-4

BATCH_SIZE = 256
GRAD_CLIP_NORM = 1.0

WEIGHT_DECAY = 1e-3
NUM_WORKERS = 0

N_FOLDS = 5
N_CONFIGS = 8

# Do not change the search configurations after seeing
# cross-validation results.

SEARCH_CONFIGS = [
    {
        "id": "C01",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 1e-3,
    },
    {
        "id": "C02",
        "hidden_dim": 32,
        "num_layers": 2,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 1e-3,
    },
    {
        "id": "C03",
        "hidden_dim": 128,
        "num_layers": 2,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 1e-3,
    },
    {
        "id": "C04",
        "hidden_dim": 64,
        "num_layers": 1,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 1e-3,
    },
    {
        "id": "C05",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.10,
        "dense_dim": 32,
        "learning_rate": 1e-3,
    },
    {
        "id": "C06",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.35,
        "dense_dim": 32,
        "learning_rate": 1e-3,
    },
    {
        "id": "C07",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 3e-4,
    },
    {
        "id": "C08",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 2e-3,
    },
]

assert len(SEARCH_CONFIGS) == N_CONFIGS
assert len(set(c["id"] for c in SEARCH_CONFIGS)) == N_CONFIGS

DEVICE_CV = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if DEVICE_CV.type == "cpu":
    torch.set_num_threads(
        min(4, os.cpu_count() or 2)
    )

# Reproducibility settings.
torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True

CV_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "LSTM_V2_Horizon"
    / "train_cv_selection_GPU"
)

CV_ROOT.mkdir(parents=True, exist_ok=True)

print("Experiment directory:", CV_ROOT)
print("Execution device:", DEVICE_CV)
print("PyTorch version:", torch.__version__)


print("=" * 105)
print("LSTM V2 — FIVE-FOLD MODEL SELECTION")
print("=" * 105)

print("Device:", DEVICE_CV)
print("PyTorch:", torch.__version__)
print("Configurations:", N_CONFIGS)
print("Folds per configuration:", N_FOLDS)
print("Maximum epochs:", MAX_EPOCHS)
print("Early-stopping patience:", PATIENCE)
print("Batch size:", BATCH_SIZE)
print("Selection metric: MEAN(H5 AP, H10 AP)")
print("Checkpoint selection: MINIMUM HOLDOUT NLL")
print("Results directory:", CV_ROOT)

if DEVICE_CV.type == "cpu":
    print(
        "\nCPU training is supported, but this is a larger "
        "experiment. A GPU is preferable if available."
    )

# ==============================================================================================
# 8C — PROTECT DATASET AND TRAINING PROTOCOL
# ==============================================================================================

def hash_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


def hash_object(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


def save_json_atomic(path, data):
    temporary = path.with_suffix(path.suffix + ".tmp")

    temporary.write_text(
        json.dumps(
            data,
            indent=2,
            sort_keys=True,
            allow_nan=False
        ),
        encoding="utf-8"
    )

    os.replace(temporary, path)


MANIFEST_PATH = (
    FREEZE_DIR
    / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)

LOCK_PATH = (
    FREEZE_DIR
    / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

release_manifest = json.loads(
    MANIFEST_PATH.read_text(encoding="utf-8")
)

freeze_lock = json.loads(
    LOCK_PATH.read_text(encoding="utf-8")
)

assert release_manifest["release_version"] == "2.0.0"

assert hash_object(release_manifest) == (
    freeze_lock["manifest_sha256_canonical"]
)

# Recheck the exact frozen TRAIN artifact.
train_path = DATA_DIR / "windows_W10_TRAIN.npz"

expected_train_hash = (
    release_manifest["data_integrity"]["file_sha256"]
    ["windows_W10_TRAIN.npz"]
)

assert hash_file(train_path) == expected_train_hash

fold_manifest = []

for fold_id, (fit_idx, hold_idx) in enumerate(
    CV_FOLDS,
    start=1
):
    fit_groups = set(CV_GROUPS[fit_idx])
    hold_groups = set(CV_GROUPS[hold_idx])

    fit_episodes = set(EPISODE_ID[fit_idx])
    hold_episodes = set(EPISODE_ID[hold_idx])

    assert fit_groups.isdisjoint(hold_groups)
    assert fit_episodes.isdisjoint(hold_episodes)

    assert len(fit_idx) > 0 and len(hold_idx) > 0

    for horizon_col in range(3):
        assert len(np.unique(Y_HORIZONS[fit_idx, horizon_col])) == 2
        assert len(np.unique(Y_HORIZONS[hold_idx, horizon_col])) == 2

    fold_manifest.append({
        "fold": fold_id,
        "fit_groups": sorted(fit_groups),
        "holdout_groups": sorted(hold_groups),
        "fit_rows": int(len(fit_idx)),
        "holdout_rows": int(len(hold_idx))
    })

PROTOCOL = {
    "dataset": "Synthetic_NAF_v2",
    "release_version": "2.0.0",
    "release_manifest_sha256": hash_object(release_manifest),
    "train_npz_sha256": expected_train_hash,
    "model_type": "MONOTONIC_LSTM_FOUR_INTERVAL_NLL",
    "number_of_configs": N_CONFIGS,
    "number_of_folds": N_FOLDS,
    "search_configs": SEARCH_CONFIGS,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "minimum_nll_improvement": MIN_DELTA,
    "batch_size": BATCH_SIZE,
    "weight_decay": WEIGHT_DECAY,
    "gradient_clip_norm": GRAD_CLIP_NORM,
    "seed": SEARCH_SEED,
    "folds": fold_manifest,
    "selection_metric": "MEAN_OF_FOLD_H5_AND_H10_AP",
    "checkpoint_rule": "BEST_FOLD_HOLDOUT_NLL",
    "epoch_rule": "MEDIAN_BEST_EPOCH_ACROSS_FIVE_FOLDS",
    "feature_names": FEATURES,
    "preprocessing": {
        "method": "LOG1P_MEDIAN_IQR",
        "clip_value": 8.0,
        "fitted_on": "FOLD_FITTING_WINDOWS_ONLY"
    },
    "device": str(DEVICE_CV),
    "torch_version": torch.__version__,
}

PROTOCOL_HASH = hash_object(PROTOCOL)

protocol_path = CV_ROOT / "SEARCH_PROTOCOL.json"

if protocol_path.exists():
    saved_protocol = json.loads(
        protocol_path.read_text(encoding="utf-8")
    )

    if saved_protocol != PROTOCOL:
        raise RuntimeError(
            "Existing LSTM CV results use a different protocol. "
            "Do not mix experiments. Inspect the existing records."
        )
else:
    save_json_atomic(protocol_path, PROTOCOL)

print("\n✓ Frozen release manifest verified")
print("✓ Frozen TRAIN artifact verified")
print("✓ All five folds are group-disjoint")
print("✓ Search protocol fingerprint:", PROTOCOL_HASH[:20])

# ==============================================================================================
# 8D — MODEL, PREPROCESSING AND SEED HELPERS
# ==============================================================================================

def set_cv_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_cv_model(config):

    return LSTMHorizonForecaster(
        input_dim=14,
        hidden_dim=int(config["hidden_dim"]),
        num_layers=int(config["num_layers"]),
        dropout=float(config["dropout"]),
        dense_dim=int(config["dense_dim"])
    ).to(DEVICE_CV)


def create_fold_loaders(fit_idx, hold_idx, seed):

    # New scaler for every fold. No leakage from holdout.
    scaler = TemporalRobustScaler(
        clip_value=8.0
    ).fit(X_RAW[fit_idx])

    x_fit = scaler.transform(X_RAW[fit_idx])
    x_hold = scaler.transform(X_RAW[hold_idx])

    y_fit = Y_HORIZONS[fit_idx].astype(np.float32)
    y_hold = Y_HORIZONS[hold_idx].astype(np.float32)

    fit_dataset = TensorDataset(
        torch.from_numpy(x_fit),
        torch.from_numpy(y_fit)
    )

    hold_dataset = TensorDataset(
        torch.from_numpy(x_hold),
        torch.from_numpy(y_hold)
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    fit_loader = DataLoader(
        fit_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE_CV.type == "cuda")
    )

    hold_loader = DataLoader(
        hold_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE_CV.type == "cuda")
    )

    return fit_loader, hold_loader


# ==============================================================================================
# 8E — TRAINING FUNCTION
# ==============================================================================================

def train_cv_epoch(model, loader, optimizer):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for xb, yb in loader:

        xb = xb.to(
            DEVICE_CV,
            non_blocking=True
        )

        yb = yb.to(
            DEVICE_CV,
            non_blocking=True
        )

        target_class = horizon_labels_to_interval_class(
            yb
        )

        optimizer.zero_grad(set_to_none=True)

        output = model(xb)

        loss = interval_nll_from_logits(
            output["conditional_logits"],
            target_class
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Nonfinite training loss."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_NORM,
            error_if_nonfinite=True
        )

        optimizer.step()

        size = len(xb)

        total_loss += float(loss.item()) * size
        total_samples += size

    return total_loss / total_samples


# ==============================================================================================
# 8F — FOLD-HOLDOUT EVALUATION
# ==============================================================================================

@torch.inference_mode()
def evaluate_cv_epoch(model, loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    collected_labels = []
    collected_probabilities = []

    for xb, yb in loader:

        xb = xb.to(
            DEVICE_CV,
            non_blocking=True
        )

        y_device = yb.to(
            DEVICE_CV,
            non_blocking=True
        )

        target_class = horizon_labels_to_interval_class(
            y_device
        )

        output = model(xb)

        loss = interval_nll_from_logits(
            output["conditional_logits"],
            target_class
        )

        probability = output["probabilities"]

        if not torch.isfinite(probability).all():
            raise RuntimeError(
                "Nonfinite CV probabilities."
            )

        if not (
            torch.all(probability[:, 0] <= probability[:, 1] + 1e-6)
            and torch.all(probability[:, 1] <= probability[:, 2] + 1e-6)
        ):
            raise RuntimeError(
                "Horizon monotonicity violated."
            )

        collected_labels.append(yb.numpy())

        collected_probabilities.append(
            probability.cpu().numpy()
        )

        size = len(xb)

        total_loss += float(loss.item()) * size
        total_samples += size

    y = np.concatenate(collected_labels)
    p = np.concatenate(collected_probabilities)

    metrics = {
        "holdout_nll": float(
            total_loss / total_samples
        )
    }

    for col, horizon in enumerate([5, 10, 15]):

        metrics[f"H{horizon}_AP"] = float(
            average_precision_score(
                y[:, col],
                p[:, col]
            )
        )

        metrics[f"H{horizon}_ROC_AUC"] = float(
            roc_auc_score(
                y[:, col],
                p[:, col]
            )
        )

        metrics[f"H{horizon}_Brier"] = float(
            brier_score_loss(
                y[:, col],
                p[:, col]
            )
        )

    metrics["H5_H10_Mean_AP"] = float(
        (
            metrics["H5_AP"]
            + metrics["H10_AP"]
        ) / 2
    )

    for key, value in metrics.items():
        if not np.isfinite(value):
            raise RuntimeError(
                f"Nonfinite fold metric: {key}"
            )

    return metrics


# ==============================================================================================
# 8G — TRAIN ONE CONFIGURATION ON ONE FOLD
# ==============================================================================================

def fit_one_cv_fold(config, config_index, fold_id):

    fit_idx, hold_idx = CV_FOLDS[fold_id - 1]

    # Each configuration/fold gets its own deterministic seed.
    fold_seed = (
        SEARCH_SEED
        + config_index * 100
        + fold_id
    )

    set_cv_seed(fold_seed)

    fit_loader, hold_loader = create_fold_loaders(
        fit_idx,
        hold_idx,
        fold_seed
    )

    model = build_cv_model(config)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(config["learning_rate"]),
        weight_decay=WEIGHT_DECAY
    )

    best_nll = float("inf")
    best_epoch = None
    best_metrics = None

    bad_epochs = 0
    history = []

    start = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):

        train_nll = train_cv_epoch(
            model,
            fit_loader,
            optimizer
        )

        hold_metrics = evaluate_cv_epoch(
            model,
            hold_loader
        )

        row = {
            "epoch": epoch,
            "train_nll": float(train_nll),
            **hold_metrics
        }

        history.append(row)

        # Fixed checkpoint-selection rule:
        # minimum fold-holdout NLL, not best AP.
        if (
            hold_metrics["holdout_nll"]
            < best_nll - MIN_DELTA
        ):

            best_nll = hold_metrics["holdout_nll"]
            best_epoch = epoch
            best_metrics = dict(hold_metrics)

            bad_epochs = 0

        else:
            bad_epochs += 1

        print(
            f"{config['id']} | "
            f"Fold {fold_id} | "
            f"Epoch {epoch:02d} | "
            f"Train NLL={train_nll:.5f} | "
            f"Hold NLL={hold_metrics['holdout_nll']:.5f} | "
            f"H5 AP={hold_metrics['H5_AP']:.5f} | "
            f"H10 AP={hold_metrics['H10_AP']:.5f}"
        )

        if bad_epochs >= PATIENCE:
            print(
                f"  Early stopping: best epoch {best_epoch}, "
                f"best holdout NLL {best_nll:.5f}"
            )
            break

    if best_epoch is None:
        raise RuntimeError(
            f"{config['id']} fold {fold_id}: "
            "no valid checkpoint."
        )

    result = {
        "config_id": config["id"],
        "fold": fold_id,
        "seed": fold_seed,
        "best_epoch": int(best_epoch),
        "best_holdout_nll": float(best_nll),
        "selected_metrics": best_metrics,
        "epochs_completed": len(history),
        "elapsed_seconds": float(
            time.perf_counter() - start
        ),
        "fit_windows": int(len(fit_idx)),
        "holdout_windows": int(len(hold_idx)),
        "history": history,
        "protocol_hash": PROTOCOL_HASH,
    }

    del model, optimizer
    del fit_loader, hold_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ==============================================================================================
# 8H — RUN OR RESUME ALL 40 FOLD EXPERIMENTS
# ==============================================================================================

ALL_FOLD_RESULTS = []

for config_index, config in enumerate(
    SEARCH_CONFIGS,
    start=1
):

    config_id = config["id"]

    print("\n" + "=" * 105)
    print(
        f"CONFIGURATION {config_index}/{N_CONFIGS}: {config_id}"
    )
    print("=" * 105)

    print(config)

    config_directory = CV_ROOT / config_id

    config_directory.mkdir(
        parents=True,
        exist_ok=True
    )

    for fold_id in range(1, N_FOLDS + 1):

        result_path = (
            config_directory
            / f"FOLD_{fold_id:02d}_RESULT.json"
        )

        if result_path.exists():

            result = json.loads(
                result_path.read_text(
                    encoding="utf-8"
                )
            )

            if result.get("protocol_hash") != PROTOCOL_HASH:
                raise RuntimeError(
                    "Existing fold result was generated "
                    "with a different protocol."
                )

            if (
                result.get("config_id") != config_id
                or result.get("fold") != fold_id
            ):
                raise RuntimeError(
                    "Existing fold identity mismatch."
                )

            print(
                f"{config_id} / Fold {fold_id}: "
                "existing completed result loaded"
            )

        else:

            result = fit_one_cv_fold(
                config=config,
                config_index=config_index,
                fold_id=fold_id
            )

            save_json_atomic(
                result_path,
                result
            )

            print(
                f"✓ {config_id} / Fold {fold_id} "
                "completed and saved"
            )

        ALL_FOLD_RESULTS.append(result)

# ==============================================================================================
# 8I — SELECT CONFIGURATION FROM TRAIN-ONLY CV
# ==============================================================================================

summary_rows = []

for config in SEARCH_CONFIGS:

    matches = [
        r for r in ALL_FOLD_RESULTS
        if r["config_id"] == config["id"]
    ]

    if len(matches) != N_FOLDS:
        raise RuntimeError(
            f"Missing CV folds for {config['id']}"
        )

    best_epochs = [
        r["best_epoch"]
        for r in matches
    ]

    h5_ap = [
        r["selected_metrics"]["H5_AP"]
        for r in matches
    ]

    h10_ap = [
        r["selected_metrics"]["H10_AP"]
        for r in matches
    ]

    h15_ap = [
        r["selected_metrics"]["H15_AP"]
        for r in matches
    ]

    hold_nll = [
        r["best_holdout_nll"]
        for r in matches
    ]

    selection_scores = [
        (
            r["selected_metrics"]["H5_AP"]
            +
            r["selected_metrics"]["H10_AP"]
        ) / 2
        for r in matches
    ]

    # Five folds -> median is one observed integer epoch.
    final_epochs = int(
        np.median(best_epochs)
    )

    summary_rows.append({
        "Config_ID": config["id"],
        "Mean_H5_AP": float(np.mean(h5_ap)),
        "Mean_H10_AP": float(np.mean(h10_ap)),
        "Mean_H15_AP": float(np.mean(h15_ap)),
        "Mean_H5_H10_AP": float(
            np.mean(selection_scores)
        ),
        "SD_H5_H10_AP": float(
            np.std(selection_scores, ddof=1)
        ),
        "Mean_Holdout_NLL": float(
            np.mean(hold_nll)
        ),
        "Selected_Final_Epochs": final_epochs,
        "Fold_Best_Epochs": best_epochs,
    })

SUMMARY_DF = pd.DataFrame(
    summary_rows
).sort_values(
    ["Mean_H5_H10_AP", "Config_ID"],
    ascending=[False, True]
).reset_index(drop=True)

winner_id = str(
    SUMMARY_DF.iloc[0]["Config_ID"]
)

winner_config = next(
    c for c in SEARCH_CONFIGS
    if c["id"] == winner_id
)

winner_row = SUMMARY_DF.iloc[0]

winner_epochs = int(
    winner_row["Selected_Final_Epochs"]
)

SELECTION_RECORD = {
    "dataset": "Synthetic_NAF_v2",
    "release_version": "2.0.0",
    "protocol_hash": PROTOCOL_HASH,
    "selection_rule": "MAXIMUM_MEAN_H5_H10_AP",
    "checkpoint_rule": "MINIMUM_FOLD_HOLDOUT_NLL",
    "selected_config_id": winner_id,
    "selected_model_config": winner_config,
    "selected_final_epochs": winner_epochs,
    "selected_mean_H5_AP": float(
        winner_row["Mean_H5_AP"]
    ),
    "selected_mean_H10_AP": float(
        winner_row["Mean_H10_AP"]
    ),
    "selected_mean_H15_AP": float(
        winner_row["Mean_H15_AP"]
    ),
    "selected_mean_H5_H10_AP": float(
        winner_row["Mean_H5_H10_AP"]
    ),
    "selected_mean_holdout_nll": float(
        winner_row["Mean_Holdout_NLL"]
    ),
    "final_training_performed": False,
    "official_validation_used": False,
    "official_test_used": False,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

# Never silently overwrite a previous selected protocol.
selection_path = (
    CV_ROOT / "SELECTED_LSTM_CONFIGURATION.json"
)

if selection_path.exists():
    existing_selection = json.loads(
        selection_path.read_text(encoding="utf-8")
    )

    stable_keys = [
        "protocol_hash",
        "selected_config_id",
        "selected_final_epochs"
    ]

    for key in stable_keys:
        if (
            existing_selection.get(key)
            != SELECTION_RECORD.get(key)
        ):
            raise RuntimeError(
                "An earlier saved model selection differs. "
                "Do not silently replace the selection."
            )
else:
    save_json_atomic(
        selection_path,
        SELECTION_RECORD
    )

SUMMARY_DF.to_csv(
    CV_ROOT / "ALL_CONFIGURATIONS_CV_SUMMARY.csv",
    index=False
)

pd.DataFrame([
    {
        "Config_ID": r["config_id"],
        "Fold": r["fold"],
        "Best_Epoch": r["best_epoch"],
        "Best_Holdout_NLL": r["best_holdout_nll"],
        "H5_AP": r["selected_metrics"]["H5_AP"],
        "H10_AP": r["selected_metrics"]["H10_AP"],
        "H15_AP": r["selected_metrics"]["H15_AP"],
        "Epochs_Completed": r["epochs_completed"]
    }
    for r in ALL_FOLD_RESULTS
]).to_csv(
    CV_ROOT / "ALL_FOLD_RESULTS.csv",
    index=False
)

# ==============================================================================================
# 8J — FINAL CONSOLE REPORT
# ==============================================================================================

print("\n" + "=" * 105)
print("LSTM V2 — FIVE-FOLD MODEL SELECTION COMPLETE")
print("=" * 105)

print("\nALL CONFIGURATION RESULTS")

print(
    SUMMARY_DF[
        [
            "Config_ID",
            "Mean_H5_AP",
            "Mean_H10_AP",
            "Mean_H15_AP",
            "Mean_H5_H10_AP",
            "SD_H5_H10_AP",
            "Mean_Holdout_NLL",
            "Selected_Final_Epochs",
        ]
    ].round(6).to_string(index=False)
)

print("\nSELECTED CONFIGURATION")
print(json.dumps(
    SELECTION_RECORD,
    indent=2
))

print("\nCompleted fold experiments:", len(ALL_FOLD_RESULTS))
print("Results directory:", CV_ROOT)

print("\nFinal full-TRAIN LSTM: NOT TRAINED")
print("Official VALIDATION: NOT ACCESSED")
print("Official TEST: NOT ACCESSED")
print("Frozen dataset: UNCHANGED")

print("\nNEXT: Review selected configuration before final fitting.")


In [ ]:

# ==========================================================================================
# LSTM V2 — CELL 9
# FINAL FULL-TRAIN MODEL FITTING — SELECTED CONFIGURATION C05
#
# WARNING: RUNNING THIS CELL TRAINS THE FINAL LSTM.
#
# Uses:
#   Frozen Synthetic_NAF_v2 version 2.0.0
#   Complete TRAIN partition — 88,186 windows
#   Selected C05 configuration from GPU five-fold CV
#   Exactly 12 epochs
#   Four-interval negative log-likelihood
#
# DOES NOT:
#   - Run any hyperparameter search
#   - Perform early stopping
#   - Load official VALIDATION or TEST
#   - Modify the frozen dataset
#   - Retrain any XGBoost model
# ==========================================================================================

import os
import gc
import json
import random
import hashlib
import platform
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from torch.utils.data import TensorDataset, DataLoader

# ==========================================================================================
# 9A — PATHS
# ==========================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATA_DIR = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "release_candidate_corrected"
)

FREEZE_DIR = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "FINAL_RELEASE_FREEZE"
)

CV_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "LSTM_V2_Horizon"
    / "train_cv_selection_GPU"
)

FINAL_DIR = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "LSTM_V2_Horizon"
    / "final_train_model"
)

TRAIN_PATH = DATA_DIR / "windows_W10_TRAIN.npz"

SELECTION_PATH = (
    CV_ROOT / "SELECTED_LSTM_CONFIGURATION.json"
)

PROTOCOL_PATH = (
    CV_ROOT / "SEARCH_PROTOCOL.json"
)

MANIFEST_PATH = (
    FREEZE_DIR / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)

LOCK_PATH = (
    FREEZE_DIR / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

# Never silently overwrite an existing final model.
if FINAL_DIR.exists() and any(FINAL_DIR.iterdir()):
    raise RuntimeError(
        "The final LSTM model directory already contains files:\n"
        f"{FINAL_DIR}\n"
        "Do not overwrite an existing final-model release."
    )

# ==========================================================================================
# 9B — REQUIRED NOTEBOOK VARIABLES
# ==========================================================================================

REQUIRED = [
    "X_RAW",
    "Y_HORIZONS",
    "EPISODE_ID",
    "CV_GROUPS",
    "FEATURES",
    "TemporalRobustScaler",
    "LSTMHorizonForecaster",
    "horizon_labels_to_interval_class",
    "interval_nll_from_logits"
]

missing = [
    name for name in REQUIRED
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run preparation Cells 1–6 first.\n"
        f"Missing: {missing}"
    )

assert X_RAW.shape == (88186, 10, 14)
assert Y_HORIZONS.shape == (88186, 3)
assert len(FEATURES) == 14
assert len(np.unique(EPISODE_ID)) == 1400
assert len(np.unique(CV_GROUPS)) == 35

assert np.isfinite(X_RAW).all()
assert np.all(X_RAW >= 0)

assert np.all(Y_HORIZONS[:, 0] <= Y_HORIZONS[:, 1])
assert np.all(Y_HORIZONS[:, 1] <= Y_HORIZONS[:, 2])

# ==========================================================================================
# 9C — LOAD SAVED CONFIGURATION
# ==========================================================================================

def load_json(path):
    if not path.is_file():
        raise FileNotFoundError(path)

    return json.loads(
        path.read_text(encoding="utf-8")
    )


def hash_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def hash_json(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


def save_json(path, value):
    path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            allow_nan=False
        ),
        encoding="utf-8"
    )


selection = load_json(SELECTION_PATH)
protocol = load_json(PROTOCOL_PATH)
manifest = load_json(MANIFEST_PATH)
freeze_lock = load_json(LOCK_PATH)

assert manifest["release_version"] == "2.0.0"

assert hash_json(manifest) == (
    freeze_lock["manifest_sha256_canonical"]
)

assert selection["protocol_hash"] == hash_json(protocol)

assert selection["selected_config_id"] == "C05"

assert selection["selected_final_epochs"] == 12

assert selection["final_training_performed"] is False

assert selection["official_validation_used"] is False
assert selection["official_test_used"] is False

assert protocol["number_of_configs"] == 8
assert protocol["number_of_folds"] == 5

assert protocol["selection_metric"] == (
    "MEAN_OF_FOLD_H5_AND_H10_AP"
)

assert protocol["checkpoint_rule"] == (
    "BEST_FOLD_HOLDOUT_NLL"
)

assert protocol["feature_names"] == FEATURES

selected_config = selection["selected_model_config"]

expected_config = {
    "id": "C05",
    "hidden_dim": 64,
    "num_layers": 2,
    "dropout": 0.1,
    "dense_dim": 32,
    "learning_rate": 0.001
}

if selected_config != expected_config:
    raise RuntimeError(
        "Selected C05 configuration differs from expected values."
    )

# Verify all 40 GPU fold results actually exist.
for config_id in [
    f"C{i:02d}" for i in range(1, 9)
]:
    for fold in range(1, 6):

        path = (
            CV_ROOT
            / config_id
            / f"FOLD_{fold:02d}_RESULT.json"
        )

        result = load_json(path)

        if result["protocol_hash"] != selection["protocol_hash"]:
            raise RuntimeError(
                f"CV protocol mismatch: {path}"
            )

        if result["config_id"] != config_id:
            raise RuntimeError(
                f"Configuration mismatch: {path}"
            )

        if result["fold"] != fold:
            raise RuntimeError(
                f"Fold mismatch: {path}"
            )

print("✓ Selected C05 configuration verified")
print("✓ All 40 GPU CV fold results verified")

# ==========================================================================================
# 9D — VERIFY FROZEN TRAIN DATA
# ==========================================================================================

expected_train_hash = (
    manifest["data_integrity"]["file_sha256"]
    ["windows_W10_TRAIN.npz"]
)

if hash_file(TRAIN_PATH) != expected_train_hash:
    raise RuntimeError(
        "Frozen TRAIN artifact integrity failure."
    )

with np.load(TRAIN_PATH, allow_pickle=False) as pack:

    saved_x = pack["X"]
    saved_y = pack["y_horizons"]

    if not np.array_equal(
        X_RAW,
        saved_x
    ):
        raise RuntimeError(
            "Current TRAIN features differ from frozen data."
        )

    if not np.array_equal(
        Y_HORIZONS,
        saved_y
    ):
        raise RuntimeError(
            "Current TRAIN labels differ from frozen data."
        )

print("✓ Frozen TRAIN arrays verified")

# ==========================================================================================
# 9E — FIXED FINAL TRAINING SETTINGS
# ==========================================================================================

FINAL_SEED = 20261022

EPOCHS = 12
BATCH_SIZE = 256

LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-3

GRAD_CLIP = 1.0

NUM_WORKERS = 0

TRAIN_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(FINAL_SEED)
np.random.seed(FINAL_SEED)
torch.manual_seed(FINAL_SEED)

if TRAIN_DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(FINAL_SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.set_num_threads(
        min(4, os.cpu_count() or 2)
    )

print("\n" + "=" * 100)
print("LSTM V2 — FINAL FULL-TRAIN FITTING")
print("=" * 100)

print("Device:", TRAIN_DEVICE)
print("PyTorch:", torch.__version__)

if TRAIN_DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Selected configuration: C05")
print("Training epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Gradient clipping:", GRAD_CLIP)

print("\nTRAIN episodes:", len(np.unique(EPISODE_ID)))
print("TRAIN windows:", len(X_RAW))

print("\nOfficial VALIDATION: NOT LOADED")
print("Official TEST: NOT LOADED")

# ==========================================================================================
# 9F — FIT SCALER ON ALL TRAIN WINDOWS ONLY
# ==========================================================================================

FINAL_SCALER = TemporalRobustScaler(
    clip_value=8.0
)

FINAL_SCALER.fit(X_RAW)

X_TRAIN_SCALED = FINAL_SCALER.transform(
    X_RAW
)

Y_TRAIN = np.ascontiguousarray(
    Y_HORIZONS.astype(np.float32)
)

assert X_TRAIN_SCALED.shape == (88186, 10, 14)
assert np.isfinite(X_TRAIN_SCALED).all()

scaler_state = FINAL_SCALER.state_dict()

assert scaler_state["feature_names"] == FEATURES

print("\n✓ Full-TRAIN-only scaler fitted")
print("Scaled input:", X_TRAIN_SCALED.shape)

# ==========================================================================================
# 9G — CREATE TRAIN DATALOADER
# ==========================================================================================

train_dataset = TensorDataset(
    torch.from_numpy(X_TRAIN_SCALED),
    torch.from_numpy(Y_TRAIN)
)

train_generator = torch.Generator()
train_generator.manual_seed(FINAL_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=NUM_WORKERS,
    pin_memory=(TRAIN_DEVICE.type == "cuda")
)

# ==========================================================================================
# 9H — INITIALIZE A BRAND-NEW FINAL MODEL
# ==========================================================================================

final_model_config = {
    "input_dim": 14,
    "hidden_dim": int(selected_config["hidden_dim"]),
    "num_layers": int(selected_config["num_layers"]),
    "dropout": float(selected_config["dropout"]),
    "dense_dim": int(selected_config["dense_dim"])
}

FINAL_MODEL = LSTMHorizonForecaster(
    **final_model_config
).to(TRAIN_DEVICE)

FINAL_OPTIMIZER = torch.optim.AdamW(
    FINAL_MODEL.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

parameter_count = sum(
    p.numel()
    for p in FINAL_MODEL.parameters()
)

print("\nFinal model architecture:")
print(FINAL_MODEL)
print("\nTrainable parameters:", parameter_count)

# ==========================================================================================
# 9I — TRAIN FOR EXACTLY 12 EPOCHS
#
# No validation loader.
# No early stopping.
# No hyperparameter changes.
# ==========================================================================================

FINAL_HISTORY = []

start_time = time.perf_counter()

for epoch in range(1, EPOCHS + 1):

    FINAL_MODEL.train()

    running_loss = 0.0
    seen = 0

    for xb, yb in train_loader:

        xb = xb.to(
            TRAIN_DEVICE,
            non_blocking=True
        )

        yb = yb.to(
            TRAIN_DEVICE,
            non_blocking=True
        )

        interval_targets = horizon_labels_to_interval_class(
            yb
        )

        FINAL_OPTIMIZER.zero_grad(
            set_to_none=True
        )

        output = FINAL_MODEL(xb)

        loss = interval_nll_from_logits(
            output["conditional_logits"],
            interval_targets
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite loss at epoch {epoch}."
            )

        loss.backward()

        for name, parameter in FINAL_MODEL.named_parameters():

            if (
                parameter.grad is not None
                and not torch.isfinite(parameter.grad).all()
            ):
                raise RuntimeError(
                    f"Nonfinite gradient detected: {name}"
                )

        torch.nn.utils.clip_grad_norm_(
            FINAL_MODEL.parameters(),
            max_norm=GRAD_CLIP,
            error_if_nonfinite=True
        )

        FINAL_OPTIMIZER.step()

        batch_size_actual = len(xb)

        running_loss += (
            float(loss.item())
            * batch_size_actual
        )

        seen += batch_size_actual

    epoch_loss = running_loss / seen

    if not np.isfinite(epoch_loss):
        raise RuntimeError(
            f"Nonfinite epoch loss: {epoch}"
        )

    elapsed = time.perf_counter() - start_time

    row = {
        "epoch": epoch,
        "train_nll": float(epoch_loss),
        "examples_seen": int(seen),
        "elapsed_seconds": float(elapsed)
    }

    FINAL_HISTORY.append(row)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Full TRAIN NLL={epoch_loss:.6f} | "
        f"Examples={seen:,}"
    )

assert len(FINAL_HISTORY) == 12

FINAL_HISTORY_DF = pd.DataFrame(FINAL_HISTORY)

# ==========================================================================================
# 9J — EVALUATE OUTPUT VALIDITY ON A TRAIN PREVIEW
#
# This is only a numerical sanity check.
# It is NOT an official performance evaluation.
# ==========================================================================================

FINAL_MODEL.eval()

with torch.inference_mode():

    preview = torch.from_numpy(
        X_TRAIN_SCALED[:256]
    ).to(TRAIN_DEVICE)

    preview_output = FINAL_MODEL(preview)

    preview_probs = preview_output["probabilities"]

    assert preview_probs.shape == (256, 3)
    assert torch.isfinite(preview_probs).all()

    assert torch.all(
        (preview_probs >= 0)
        &
        (preview_probs <= 1)
    )

    assert torch.all(
        preview_probs[:, 0]
        <= preview_probs[:, 1] + 1e-6
    )

    assert torch.all(
        preview_probs[:, 1]
        <= preview_probs[:, 2] + 1e-6
    )

print("\n✓ Numerical output verification passed")
print("✓ Horizon monotonicity verified")
print("✓ Exactly 12 epochs completed")

# ==========================================================================================
# 9K — PREPARE FINAL MODEL RECORD
# ==========================================================================================

training_record = {
    "model_name": "LSTM_V2_MONOTONIC_HORIZON",
    "dataset": "Synthetic_NAF_v2",
    "dataset_version": "2.0.0",
    "dataset_manifest_sha256": hash_json(manifest),
    "train_data_sha256": expected_train_hash,
    "training_split": "TRAIN",
    "training_windows": int(len(X_RAW)),
    "training_episodes": int(
        len(np.unique(EPISODE_ID))
    ),
    "input_shape": [10, 14],
    "output_horizons_minutes": [5, 10, 15],
    "model_architecture": final_model_config,
    "trainable_parameters": int(parameter_count),
    "selected_configuration_id": "C05",
    "selected_configuration": selected_config,
    "selection_protocol_hash": selection["protocol_hash"],
    "selected_cv_mean_H5_AP": selection["selected_mean_H5_AP"],
    "selected_cv_mean_H10_AP": selection["selected_mean_H10_AP"],
    "selected_cv_mean_H15_AP": selection["selected_mean_H15_AP"],
    "training_epochs": 12,
    "training_seed": FINAL_SEED,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "gradient_clip_norm": GRAD_CLIP,
    "loss_function": "FOUR_INTERVAL_NEGATIVE_LOG_LIKELIHOOD",
    "preprocessing": "LOG1P_MEDIAN_IQR_CLIP8",
    "feature_names": FEATURES,
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "device": str(TRAIN_DEVICE),
    "training_status": "COMPLETED",
    "official_validation_used": False,
    "official_test_used": False,
    "dataset_modified": False,
    "trained_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

# ==========================================================================================
# 9L — SAVE MODEL, SCALER AND HISTORY
#
# The final folder is created only after training and
# numerical checks have succeeded.
# ==========================================================================================

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=False
)

MODEL_FILE = (
    FINAL_DIR
    / "LSTM_V2_MONOTONIC_HORIZON_STATE.pt"
)

SCALER_FILE = (
    FINAL_DIR
    / "LSTM_V2_FULL_TRAIN_SCALER.json"
)

HISTORY_FILE = (
    FINAL_DIR
    / "LSTM_V2_FINAL_TRAIN_HISTORY.csv"
)

RECORD_FILE = (
    FINAL_DIR
    / "LSTM_V2_FINAL_TRAIN_RECORD.json"
)

REGISTRY_FILE = (
    FINAL_DIR
    / "LSTM_V2_FINAL_MODEL_REGISTRY.json"
)

# Save plain CPU tensors rather than a serialized model object.
# Architecture is reconstructed from the JSON record.
cpu_state = {
    key: value.detach().cpu().clone()
    for key, value in FINAL_MODEL.state_dict().items()
}

torch.save(
    cpu_state,
    MODEL_FILE
)

save_json(
    SCALER_FILE,
    scaler_state
)

FINAL_HISTORY_DF.to_csv(
    HISTORY_FILE,
    index=False
)

training_record["model_sha256"] = hash_file(MODEL_FILE)
training_record["scaler_sha256"] = hash_file(SCALER_FILE)
training_record["history_sha256"] = hash_file(HISTORY_FILE)

save_json(
    RECORD_FILE,
    training_record
)

registry = {
    "dataset": "Synthetic_NAF_v2",
    "dataset_version": "2.0.0",
    "model_name": "LSTM_V2_MONOTONIC_HORIZON",
    "training_status": "FINAL_FULL_TRAIN_COMPLETED",
    "validation_status": "NOT_STARTED",
    "test_status": "EMBARGOED",
    "selected_configuration": "C05",
    "epochs": 12,
    "model_file": MODEL_FILE.name,
    "model_sha256": training_record["model_sha256"],
    "scaler_file": SCALER_FILE.name,
    "scaler_sha256": training_record["scaler_sha256"],
    "training_record_file": RECORD_FILE.name,
    "training_record_sha256": hash_file(RECORD_FILE),
    "feature_names": FEATURES,
    "horizons": [5, 10, 15]
}

save_json(
    REGISTRY_FILE,
    registry
)

# ==========================================================================================
# 9M — RELOAD VERIFICATION
# ==========================================================================================

reloaded_state = torch.load(
    MODEL_FILE,
    map_location="cpu",
    weights_only=True
)

RELOADED_MODEL = LSTMHorizonForecaster(
    **final_model_config
).cpu()

RELOADED_MODEL.load_state_dict(
    reloaded_state,
    strict=True
)

RELOADED_MODEL.eval()

FINAL_MODEL.eval()

with torch.inference_mode():

    sample_cpu = torch.from_numpy(
        X_TRAIN_SCALED[:32]
    )

    original_prediction = (
        FINAL_MODEL(
            sample_cpu.to(TRAIN_DEVICE)
        )["probabilities"]
        .cpu()
    )

    reloaded_prediction = (
        RELOADED_MODEL(
            sample_cpu
        )["probabilities"]
    )

    if not torch.allclose(
        original_prediction,
        reloaded_prediction,
        atol=1e-5,
        rtol=1e-5
    ):
        raise RuntimeError(
            "Reloaded LSTM predictions differ from original model."
        )

assert hash_file(MODEL_FILE) == registry["model_sha256"]
assert hash_file(SCALER_FILE) == registry["scaler_sha256"]

print("✓ Saved model reloaded successfully")
print("✓ Reloaded predictions match trained model")
print("✓ Model and scaler hashes verified")

# ==========================================================================================
# 9N — FINAL REPORT
# ==========================================================================================

print("\n" + "=" * 100)
print("LSTM V2 — FINAL FULL-TRAIN FITTING COMPLETE")
print("=" * 100)

print("Selected configuration: C05")
print("Training windows:", len(X_RAW))
print("Training episodes:", len(np.unique(EPISODE_ID)))
print("Epochs completed:", len(FINAL_HISTORY_DF))

print(
    "Initial epoch NLL:",
    round(
        float(FINAL_HISTORY_DF.iloc[0]["train_nll"]),
        6
    )
)

print(
    "Final epoch NLL:",
    round(
        float(FINAL_HISTORY_DF.iloc[-1]["train_nll"]),
        6
    )
)

print("\nSaved model:")
print(MODEL_FILE)

print("\nSaved scaler:")
print(SCALER_FILE)

print("\nModel SHA256:")
print(training_record["model_sha256"])

print("\nFinal LSTM model: TRAINED AND SAVED")
print("Additional hyperparameter optimization: NONE")
print("Official VALIDATION: NOT USED")
print("Official TEST: NOT USED")
print("Frozen dataset: UNCHANGED")

print("\nNEXT: Verify artifacts before official validation.")


In [ ]:

# ==========================================================================================
# LSTM V2 — CELL 10
# OFFICIAL VALIDATION OF THE FROZEN C05 MONOTONIC LSTM
#
# READ-ONLY MODEL EVALUATION:
#   - NO training
#   - NO optimizer
#   - NO backward pass
#   - NO hyperparameter changes
#   - NO probability recalibration
#   - NO official TEST access
#
# Evaluates:
#   H5, H10, H15
#   AP, ROC-AUC, Brier, ECE
#   Fixed-threshold operating metrics
#   Source-specific results
#   Benign false-alert statistics
#   Episode-cluster bootstrap confidence intervals
#   Monotonicity
#   Comparison against previously recorded XGBoost results
#
# Results are saved outside the frozen dataset.
# ==========================================================================================

import os
import json
import hashlib
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix
)

# ==========================================================================================
# 10A — PATHS AND FIXED VALIDATION SETTINGS
# ==========================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATA_DIR_10 = (
    ROOT / "Synthetic_NAF_v2/release_candidate_corrected"
)

FREEZE_DIR_10 = (
    ROOT / "Synthetic_NAF_v2/FINAL_RELEASE_FREEZE"
)

MODEL_DIR_10 = (
    ROOT / "Synthetic_NAF_v2/models/"
    "LSTM_V2_Horizon/final_train_model"
)

CV_DIR_10 = (
    ROOT / "Synthetic_NAF_v2/models/"
    "LSTM_V2_Horizon/train_cv_selection_GPU"
)

REPORT_DIR_10 = (
    ROOT / "Synthetic_NAF_v2/models/"
    "LSTM_V2_Horizon/official_validation"
)

VAL_PATH_10 = DATA_DIR_10 / "windows_W10_VALIDATION.npz"

META_PATH_10 = (
    DATA_DIR_10
    / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
)

MANIFEST_PATH_10 = (
    FREEZE_DIR_10
    / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)

LOCK_PATH_10 = (
    FREEZE_DIR_10
    / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

MODEL_PATH_10 = (
    MODEL_DIR_10
    / "LSTM_V2_MONOTONIC_HORIZON_STATE.pt"
)

SCALER_PATH_10 = (
    MODEL_DIR_10
    / "LSTM_V2_FULL_TRAIN_SCALER.json"
)

TRAIN_RECORD_PATH_10 = (
    MODEL_DIR_10
    / "LSTM_V2_FINAL_TRAIN_RECORD.json"
)

REGISTRY_PATH_10 = (
    MODEL_DIR_10
    / "LSTM_V2_FINAL_MODEL_REGISTRY.json"
)

SELECTION_PATH_10 = (
    CV_DIR_10
    / "SELECTED_LSTM_CONFIGURATION.json"
)

PROTOCOL_PATH_10 = (
    CV_DIR_10 / "SEARCH_PROTOCOL.json"
)

EXPECTED_MODEL_SHA256 = (
    "c44227e823986baae57ccd10fa70fa3d00"
    "b63501bc4ee8fe70d5666af0f6799a"
)

HORIZONS_10 = [5, 10, 15]
FIXED_THRESHOLDS_10 = [0.10, 0.20, 0.50]

BOOTSTRAP_REPLICATES_10 = 300
BOOTSTRAP_SEED_10 = 20261023

INFERENCE_BATCH_SIZE_10 = 1024

# No validation-based threshold optimization.
# No validation-based choice of final training epochs.

# Prevent accidental report overwriting before any evaluation.
if (
    REPORT_DIR_10.exists()
    and any(REPORT_DIR_10.iterdir())
):
    raise RuntimeError(
        "Official LSTM validation results already exist:\n"
        f"{REPORT_DIR_10}\n"
        "Existing reports will not be overwritten."
    )

# ==========================================================================================
# 10B — INTEGRITY HELPERS
# ==========================================================================================

def load_json_10(path):
    if not path.is_file():
        raise FileNotFoundError(path)

    return json.loads(
        path.read_text(encoding="utf-8")
    )


def sha256_file_10(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def canonical_sha256_10(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


def save_json_10(path, obj):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False
        ),
        encoding="utf-8"
    )


def parse_bool_10(series):
    """
    Strict parsing avoids the dangerous:
        astype(bool)
    behavior for string values such as "False".
    """

    normalized = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    true_values = {"true", "1", "yes"}
    false_values = {"false", "0", "no"}

    allowed = true_values | false_values

    if not normalized.isin(allowed).all():
        unexpected = normalized[
            ~normalized.isin(allowed)
        ].unique().tolist()

        raise ValueError(
            f"Unexpected boolean metadata values: {unexpected}"
        )

    return normalized.isin(true_values).to_numpy(dtype=bool)


# ==========================================================================================
# 10C — VERIFY SAVED MODEL, SCALER AND FREEZE
# ==========================================================================================

required_objects_10 = [
    "LSTMHorizonForecaster",
]

missing_10 = [
    x for x in required_objects_10
    if x not in globals()
]

if missing_10:
    raise RuntimeError(
        "Run the architecture-definition Cell 5 first. "
        f"Missing: {missing_10}"
    )

manifest_10 = load_json_10(MANIFEST_PATH_10)
lock_10 = load_json_10(LOCK_PATH_10)

registry_10 = load_json_10(REGISTRY_PATH_10)
training_record_10 = load_json_10(TRAIN_RECORD_PATH_10)
scaler_record_10 = load_json_10(SCALER_PATH_10)

selection_10 = load_json_10(SELECTION_PATH_10)
protocol_10 = load_json_10(PROTOCOL_PATH_10)

assert manifest_10["release_version"] == "2.0.0"

assert (
    canonical_sha256_10(manifest_10)
    == lock_10["manifest_sha256_canonical"]
)

assert (
    canonical_sha256_10(protocol_10)
    == selection_10["protocol_hash"]
)

assert selection_10["selected_config_id"] == "C05"
assert selection_10["selected_final_epochs"] == 12

assert registry_10["training_status"] == (
    "FINAL_FULL_TRAIN_COMPLETED"
)

assert registry_10["validation_status"] == "NOT_STARTED"
assert registry_10["test_status"] == "EMBARGOED"

assert training_record_10["training_epochs"] == 12
assert training_record_10["training_windows"] == 88186
assert training_record_10["training_episodes"] == 1400

assert training_record_10["official_validation_used"] is False
assert training_record_10["official_test_used"] is False

assert registry_10["model_sha256"] == EXPECTED_MODEL_SHA256

actual_model_hash_10 = sha256_file_10(MODEL_PATH_10)

if actual_model_hash_10 != EXPECTED_MODEL_SHA256:
    raise RuntimeError(
        "Final LSTM model SHA256 mismatch. "
        "Official validation aborted."
    )

if sha256_file_10(SCALER_PATH_10) != registry_10["scaler_sha256"]:
    raise RuntimeError("Final scaler SHA256 mismatch.")

if sha256_file_10(TRAIN_RECORD_PATH_10) != (
    registry_10["training_record_sha256"]
):
    raise RuntimeError("Training record SHA256 mismatch.")

if training_record_10["model_sha256"] != EXPECTED_MODEL_SHA256:
    raise RuntimeError("Training record model hash mismatch.")

if training_record_10["scaler_sha256"] != (
    registry_10["scaler_sha256"]
):
    raise RuntimeError("Training record scaler hash mismatch.")

# Verify VAL against the frozen dataset manifest.
expected_val_hash_10 = (
    manifest_10["data_integrity"]["file_sha256"]
    ["windows_W10_VALIDATION.npz"]
)

if sha256_file_10(VAL_PATH_10) != expected_val_hash_10:
    raise RuntimeError(
        "Official VALIDATION dataset SHA256 mismatch."
    )

expected_meta_hash_10 = (
    manifest_10["data_integrity"]["file_sha256"]
    [META_PATH_10.name]
)

if sha256_file_10(META_PATH_10) != expected_meta_hash_10:
    raise RuntimeError("Metadata integrity mismatch.")

feature_names_10 = training_record_10["feature_names"]

assert len(feature_names_10) == 14
assert feature_names_10 == registry_10["feature_names"]
assert feature_names_10 == scaler_record_10["feature_names"]

print("=" * 100)
print("LSTM V2 — OFFICIAL VALIDATION")
print("=" * 100)

print("Frozen dataset release: 2.0.0")
print("Frozen dataset hashes: VERIFIED")
print("Final model SHA256: VERIFIED")
print("Scaler and training record: VERIFIED")
print("Selected configuration: C05")
print("Training: DISABLED")
print("Official TEST: NOT ACCESSED")

# ==========================================================================================
# 10D — LOAD OFFICIAL VALIDATION ARRAYS
# ==========================================================================================

with np.load(VAL_PATH_10, allow_pickle=False) as pack:

    X_VAL_RAW_10 = pack["X"].astype(np.float32)

    Y_VAL_10 = pack["y_horizons"].astype(np.int8)

    VAL_ONSET_10 = pack["onset_class"].astype(np.int16)

    VAL_FAMILY_CLASS_10 = pack["family_class"].astype(np.int8)

    VAL_EPISODE_ID_10 = (
        pd.Series(pack["episode_id"])
        .astype(str)
        .to_numpy()
    )

    VAL_MINUTE_10 = pack["minute"].astype(np.int16)

assert X_VAL_RAW_10.shape == (18839, 10, 14)
assert Y_VAL_10.shape == (18839, 3)

assert len(np.unique(VAL_EPISODE_ID_10)) == 300

assert np.isfinite(X_VAL_RAW_10).all()
assert (X_VAL_RAW_10 >= 0).all()

assert np.all(Y_VAL_10[:, 0] <= Y_VAL_10[:, 1])
assert np.all(Y_VAL_10[:, 1] <= Y_VAL_10[:, 2])

for col, horizon in enumerate(HORIZONS_10):
    expected_labels = (
        (VAL_ONSET_10 >= 1)
        & (VAL_ONSET_10 <= horizon)
    ).astype(np.int8)

    assert np.array_equal(
        Y_VAL_10[:, col],
        expected_labels
    )

assert np.array_equal(
    VAL_ONSET_10 == 0,
    VAL_FAMILY_CLASS_10 == 0
)

metadata_10 = pd.read_csv(META_PATH_10)

metadata_10["Episode_ID"] = (
    metadata_10["Episode_ID"].astype(str)
)

assert metadata_10["Episode_ID"].is_unique

meta_indexed_10 = metadata_10.set_index("Episode_ID")

val_episode_set_10 = set(VAL_EPISODE_ID_10)

assert val_episode_set_10.issubset(
    set(meta_indexed_10.index)
)

val_meta_10 = meta_indexed_10.loc[
    VAL_EPISODE_ID_10
]

assert set(
    val_meta_10["Split"].astype(str)
) == {"VALIDATION"}

VAL_SOURCE_10 = (
    val_meta_10["Source"]
    .astype(str)
    .to_numpy()
)

VAL_FAMILY_NAME_10 = (
    val_meta_10["Family"]
    .astype(str)
    .to_numpy()
)

VAL_HARD_BENIGN_10 = parse_bool_10(
    val_meta_10["Hard_Benign"]
)

VAL_IS_BENIGN_10 = (
    VAL_FAMILY_NAME_10
    .astype(str)
    == "BENIGN"
)

# Confirm benign categories using the actual episode metadata.
unique_families_10 = set(
    np.unique(VAL_FAMILY_NAME_10)
)

print("\nVALIDATION INPUT")
print("Windows:", X_VAL_RAW_10.shape)
print("Unique episodes:", len(val_episode_set_10))
print("Sources:", sorted(set(VAL_SOURCE_10)))
print("Family labels:", sorted(unique_families_10))

for col, h in enumerate(HORIZONS_10):
    print(
        f"H{h} prevalence:",
        f"{Y_VAL_10[:, col].mean():.6f}"
    )

# ==========================================================================================
# 10E — RECONSTRUCT EXACT FINAL-TRAIN SCALER
#
# NO SCALER FITTING TAKES PLACE HERE.
# ==========================================================================================

scaler_median_10 = np.asarray(
    scaler_record_10["median"],
    dtype=np.float32
)

scaler_scale_10 = np.asarray(
    scaler_record_10["scale"],
    dtype=np.float32
)

scaler_clip_10 = float(
    scaler_record_10["clip_value"]
)

assert scaler_median_10.shape == (14,)
assert scaler_scale_10.shape == (14,)

assert np.isfinite(scaler_median_10).all()
assert np.isfinite(scaler_scale_10).all()

assert np.all(scaler_scale_10 > 0)

assert scaler_clip_10 == 8.0

def transform_validation_10(x):

    # Statistics are loaded from the final TRAIN scaler.
    z = np.log1p(
        x.astype(np.float32)
    )

    z = (
        z - scaler_median_10[None, None, :]
    ) / scaler_scale_10[None, None, :]

    z = np.clip(
        z,
        -scaler_clip_10,
        scaler_clip_10
    )

    return np.ascontiguousarray(
        z,
        dtype=np.float32
    )


X_VAL_10 = transform_validation_10(
    X_VAL_RAW_10
)

assert X_VAL_10.shape == (18839, 10, 14)
assert np.isfinite(X_VAL_10).all()

print("\n✓ Saved full-TRAIN scaler applied")
print("✓ No validation scaling parameters fitted")

# ==========================================================================================
# 10F — RELOAD THE FINAL TRAINED LSTM
# ==========================================================================================

architecture_10 = training_record_10["model_architecture"]

assert architecture_10 == {
    "input_dim": 14,
    "hidden_dim": 64,
    "num_layers": 2,
    "dropout": 0.1,
    "dense_dim": 32
}

EVAL_DEVICE_10 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

EVAL_MODEL_10 = LSTMHorizonForecaster(
    **architecture_10
).to(EVAL_DEVICE_10)

saved_weights_10 = torch.load(
    MODEL_PATH_10,
    map_location="cpu",
    weights_only=True
)

EVAL_MODEL_10.load_state_dict(
    saved_weights_10,
    strict=True
)

EVAL_MODEL_10.eval()

for parameter in EVAL_MODEL_10.parameters():
    parameter.requires_grad_(False)

assert not EVAL_MODEL_10.training
assert all(
    not p.requires_grad
    for p in EVAL_MODEL_10.parameters()
)

print("\n✓ Final LSTM weights loaded")
print("✓ All parameters frozen for evaluation")
print("Inference device:", EVAL_DEVICE_10)

# ==========================================================================================
# 10G — GENERATE VALIDATION PREDICTIONS
# ==========================================================================================

from torch.utils.data import TensorDataset, DataLoader

validation_dataset_10 = TensorDataset(
    torch.from_numpy(X_VAL_10)
)

validation_loader_10 = DataLoader(
    validation_dataset_10,
    batch_size=INFERENCE_BATCH_SIZE_10,
    shuffle=False,
    num_workers=0,
    pin_memory=(EVAL_DEVICE_10.type == "cuda")
)

probability_parts_10 = []

with torch.inference_mode():

    for (xb,) in validation_loader_10:

        xb = xb.to(
            EVAL_DEVICE_10,
            non_blocking=True
        )

        output = EVAL_MODEL_10(xb)

        p = output["probabilities"]

        assert p.shape[1] == 3

        if not torch.isfinite(p).all():
            raise RuntimeError(
                "Nonfinite LSTM validation probabilities."
            )

        probability_parts_10.append(
            p.cpu().numpy()
        )

P_VAL_10 = np.concatenate(
    probability_parts_10,
    axis=0
).astype(np.float64)

assert P_VAL_10.shape == (18839, 3)

assert np.isfinite(P_VAL_10).all()

assert np.all(P_VAL_10 >= 0)
assert np.all(P_VAL_10 <= 1)

monotonic_violations_5_10 = int(
    np.sum(
        P_VAL_10[:, 0] > P_VAL_10[:, 1] + 1e-7
    )
)

monotonic_violations_10_15 = int(
    np.sum(
        P_VAL_10[:, 1] > P_VAL_10[:, 2] + 1e-7
    )
)

if (
    monotonic_violations_5_10 > 0
    or monotonic_violations_10_15 > 0
):
    raise RuntimeError(
        "LSTM horizon monotonicity violation."
    )

print("\n✓ Validation predictions generated")
print("✓ All probabilities finite")
print("✓ Horizon monotonicity verified")
print("✓ No parameters updated")

# ==========================================================================================
# 10H — CALIBRATION AND METRIC FUNCTIONS
# ==========================================================================================

def ece_10_bins_10(y, p, n_bins=10):

    y = np.asarray(y, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)

    # Equal-width bins in [0, 1].
    bin_index = np.minimum(
        (p * n_bins).astype(int),
        n_bins - 1
    )

    rows = []
    ece = 0.0

    for b in range(n_bins):

        mask = bin_index == b
        count = int(mask.sum())

        if count:

            confidence = float(
                p[mask].mean()
            )

            observed = float(
                y[mask].mean()
            )

            ece += (
                count / len(y)
            ) * abs(confidence - observed)

        else:
            confidence = None
            observed = None

        rows.append({
            "bin": b,
            "bin_lower": b / n_bins,
            "bin_upper": (b + 1) / n_bins,
            "count": count,
            "mean_probability": confidence,
            "observed_event_rate": observed
        })

    return float(ece), rows


def metric_summary_10(y, p):

    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(p, dtype=np.float64)

    if len(np.unique(y)) != 2:
        raise ValueError(
            "Both classes required for AP/AUC metrics."
        )

    ece, bin_rows = ece_10_bins_10(y, p)

    metrics = {
        "windows": int(len(y)),
        "prevalence": float(y.mean()),
        "AP": float(
            average_precision_score(y, p)
        ),
        "ROC_AUC": float(
            roc_auc_score(y, p)
        ),
        "Brier": float(
            brier_score_loss(y, p)
        ),
        "ECE_10_bins": float(ece)
    }

    return metrics, bin_rows


# ==========================================================================================
# 10I — OVERALL HORIZON METRICS
# ==========================================================================================

overall_rows_10 = []
calibration_rows_10 = []

for col, horizon in enumerate(HORIZONS_10):

    metrics, bins = metric_summary_10(
        Y_VAL_10[:, col],
        P_VAL_10[:, col]
    )

    overall_rows_10.append({
        "Horizon": f"H{horizon}",
        "Windows": metrics["windows"],
        "Episodes": len(val_episode_set_10),
        "Prevalence": metrics["prevalence"],
        "AP": metrics["AP"],
        "ROC_AUC": metrics["ROC_AUC"],
        "Brier": metrics["Brier"],
        "ECE_10_bins": metrics["ECE_10_bins"]
    })

    for row in bins:
        calibration_rows_10.append({
            "Horizon": f"H{horizon}",
            **row
        })

OVERALL_DF_10 = pd.DataFrame(
    overall_rows_10
)

CALIBRATION_DF_10 = pd.DataFrame(
    calibration_rows_10
)

# ==========================================================================================
# 10J — SOURCE-SPECIFIC METRICS
# ==========================================================================================

source_rows_10 = []

for horizon_col, horizon in enumerate(HORIZONS_10):

    for source in sorted(set(VAL_SOURCE_10)):

        mask = VAL_SOURCE_10 == source

        metrics, _ = metric_summary_10(
            Y_VAL_10[mask, horizon_col],
            P_VAL_10[mask, horizon_col]
        )

        source_rows_10.append({
            "Horizon": f"H{horizon}",
            "Source": source,
            "Windows": metrics["windows"],
            "Episodes": int(
                len(np.unique(VAL_EPISODE_ID_10[mask]))
            ),
            "Prevalence": metrics["prevalence"],
            "AP": metrics["AP"],
            "ROC_AUC": metrics["ROC_AUC"],
            "Brier": metrics["Brier"],
            "ECE_10_bins": metrics["ECE_10_bins"]
        })

SOURCE_DF_10 = pd.DataFrame(
    source_rows_10
)

# ==========================================================================================
# 10K — FIXED THRESHOLD METRICS
# ==========================================================================================

threshold_rows_10 = []

for col, horizon in enumerate(HORIZONS_10):

    y = Y_VAL_10[:, col]
    p = P_VAL_10[:, col]

    for threshold in FIXED_THRESHOLDS_10:

        prediction = (
            p >= threshold
        ).astype(np.int8)

        tn, fp, fn, tp = confusion_matrix(
            y,
            prediction,
            labels=[0, 1]
        ).ravel()

        precision = (
            float(tp / (tp + fp))
            if tp + fp > 0
            else 0.0
        )

        recall = (
            float(tp / (tp + fn))
            if tp + fn > 0
            else 0.0
        )

        specificity = (
            float(tn / (tn + fp))
            if tn + fp > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        threshold_rows_10.append({
            "Horizon": f"H{horizon}",
            "Threshold": threshold,
            "TP": int(tp),
            "FP": int(fp),
            "TN": int(tn),
            "FN": int(fn),
            "Precision": precision,
            "Recall": recall,
            "Specificity": specificity,
            "F1": f1,
            "False_Positive_Rate": float(
                fp / (fp + tn)
            ) if fp + tn else 0.0
        })

THRESHOLD_DF_10 = pd.DataFrame(
    threshold_rows_10
)

# ==========================================================================================
# 10L — BENIGN FALSE-ALERT STATISTICS
#
# IMPORTANT:
#   These counts include ONLY wholly benign episodes.
#
#   The overall FP count above ALSO includes negative-horizon
#   windows from attack episodes, so those totals differ.
# ==========================================================================================

benign_rows_10 = []

for col, horizon in enumerate(HORIZONS_10):

    for threshold in FIXED_THRESHOLDS_10:

        alerted = (
            P_VAL_10[:, col] >= threshold
        )

        benign_categories = {
            "HARD_BENIGN": (
                VAL_IS_BENIGN_10
                & VAL_HARD_BENIGN_10
            ),
            "ORDINARY_BENIGN": (
                VAL_IS_BENIGN_10
                & ~VAL_HARD_BENIGN_10
            ),
            "ALL_BENIGN": VAL_IS_BENIGN_10
        }

        for category, mask in benign_categories.items():

            episode_ids = np.unique(
                VAL_EPISODE_ID_10[mask]
            )

            alerted_episodes = np.unique(
                VAL_EPISODE_ID_10[
                    mask & alerted
                ]
            )

            alert_windows = int(
                np.sum(mask & alerted)
            )

            benign_rows_10.append({
                "Horizon": f"H{horizon}",
                "Threshold": threshold,
                "Benign_Type": category,
                "Total_Episodes": int(
                    len(episode_ids)
                ),
                "Episodes_With_Any_Alert": int(
                    len(alerted_episodes)
                ),
                "False_Alert_Windows": alert_windows,
                "Fraction_Episodes_Alerted": (
                    float(
                        len(alerted_episodes)
                        / len(episode_ids)
                    )
                    if len(episode_ids)
                    else 0.0
                )
            })

BENIGN_DF_10 = pd.DataFrame(
    benign_rows_10
)

# ==========================================================================================
# 10M — EPISODE-CLUSTER BOOTSTRAP CONFIDENCE INTERVALS
#
# Resampling unit: Episode_ID.
# This accounts for repeated windows within episodes.
#
# Limitation:
#   It does not fully account for dependence among episodes
#   sharing the same simulator calibration group.
# ==========================================================================================

rng_10 = np.random.default_rng(
    BOOTSTRAP_SEED_10
)

unique_episodes_10 = np.unique(
    VAL_EPISODE_ID_10
)

episode_indices_10 = [
    np.flatnonzero(
        VAL_EPISODE_ID_10 == eid
    )
    for eid in unique_episodes_10
]

bootstrap_results_10 = {
    f"H{h}": {
        "AP": [],
        "ROC_AUC": [],
        "Brier": []
    }
    for h in HORIZONS_10
}

successful_bootstraps_10 = 0

for replicate in range(
    BOOTSTRAP_REPLICATES_10
):

    sampled_episode_positions = rng_10.integers(
        low=0,
        high=len(unique_episodes_10),
        size=len(unique_episodes_10)
    )

    sampled_indices = np.concatenate([
        episode_indices_10[position]
        for position in sampled_episode_positions
    ])

    # Preserve one common bootstrap sample for all horizons.
    valid_sample = all(
        len(
            np.unique(
                Y_VAL_10[sampled_indices, col]
            )
        ) == 2
        for col in range(3)
    )

    if not valid_sample:
        continue

    successful_bootstraps_10 += 1

    for col, h in enumerate(HORIZONS_10):

        y = Y_VAL_10[sampled_indices, col]
        p = P_VAL_10[sampled_indices, col]

        bootstrap_results_10[
            f"H{h}"
        ]["AP"].append(
            float(
                average_precision_score(y, p)
            )
        )

        bootstrap_results_10[
            f"H{h}"
        ]["ROC_AUC"].append(
            float(
                roc_auc_score(y, p)
            )
        )

        bootstrap_results_10[
            f"H{h}"
        ]["Brier"].append(
            float(
                brier_score_loss(y, p)
            )
        )

if successful_bootstraps_10 < 250:
    raise RuntimeError(
        "Insufficient successful episode-bootstrap samples."
    )

BOOTSTRAP_CI_10 = {}

for h in HORIZONS_10:

    horizon_key = f"H{h}"

    BOOTSTRAP_CI_10[horizon_key] = {}

    for metric in [
        "AP",
        "ROC_AUC",
        "Brier"
    ]:

        values = np.asarray(
            bootstrap_results_10[horizon_key][metric],
            dtype=np.float64
        )

        lower, upper = np.percentile(
            values,
            [2.5, 97.5]
        )

        BOOTSTRAP_CI_10[horizon_key][metric] = {
            "lower_95": float(lower),
            "upper_95": float(upper),
            "replicates": int(len(values))
        }

# ==========================================================================================
# 10N — XGBOOST REFERENCE COMPARISON
#
# These are the previously recorded, frozen XGBoost V2
# official validation AP scores from this research project.
#
# This section DOES NOT reload or modify XGBoost models.
# It compares metrics from the SAME official validation split.
# ==========================================================================================

XGBOOST_REFERENCE_10 = {
    "H5": {
        "AP": 0.676304,
        "ROC_AUC": 0.973142,
        "Brier": 0.031031
    },
    "H10": {
        "AP": 0.875130,
        "ROC_AUC": 0.979812,
        "Brier": 0.036030
    }
}

comparison_rows_10 = []

for horizon in ["H5", "H10"]:

    lstm_row = OVERALL_DF_10.loc[
        OVERALL_DF_10["Horizon"] == horizon
    ].iloc[0]

    xgb = XGBOOST_REFERENCE_10[horizon]

    comparison_rows_10.append({
        "Horizon": horizon,
        "LSTM_AP": float(lstm_row["AP"]),
        "XGBoost_AP": xgb["AP"],
        "AP_Difference_LSTM_Minus_XGBoost": float(
            lstm_row["AP"] - xgb["AP"]
        ),
        "LSTM_ROC_AUC": float(lstm_row["ROC_AUC"]),
        "XGBoost_ROC_AUC": xgb["ROC_AUC"],
        "LSTM_Brier": float(lstm_row["Brier"]),
        "XGBoost_Brier": xgb["Brier"]
    })

COMPARISON_DF_10 = pd.DataFrame(
    comparison_rows_10
)

# ==========================================================================================
# 10O — BUILD WINDOW-LEVEL VALIDATION PREDICTIONS TABLE
# ==========================================================================================

PREDICTIONS_DF_10 = pd.DataFrame({
    "Episode_ID": VAL_EPISODE_ID_10,
    "Minute": VAL_MINUTE_10,
    "Source": VAL_SOURCE_10,
    "Family_Metadata_Audit_Only": VAL_FAMILY_NAME_10,
    "Hard_Benign_Audit_Only": VAL_HARD_BENIGN_10,
    "Onset_Class_True": VAL_ONSET_10,
    "Family_Class_True": VAL_FAMILY_CLASS_10,
    "Y_H5": Y_VAL_10[:, 0],
    "Y_H10": Y_VAL_10[:, 1],
    "Y_H15": Y_VAL_10[:, 2],
    "P_H5": P_VAL_10[:, 0],
    "P_H10": P_VAL_10[:, 1],
    "P_H15": P_VAL_10[:, 2]
})

assert len(PREDICTIONS_DF_10) == 18839

assert not PREDICTIONS_DF_10.duplicated(
    ["Episode_ID", "Minute"]
).any()

# ==========================================================================================
# 10P — SAVE OFFICIAL VALIDATION ARTIFACTS
#
# We only create the report directory once all computations
# and integrity checks have succeeded.
#
# Existing official validation artifacts will not be replaced.
# ==========================================================================================

report_10 = {
    "report_type": "LSTM_V2_OFFICIAL_VALIDATION",
    "dataset": "Synthetic_NAF_v2",
    "dataset_version": "2.0.0",
    "model_name": "LSTM_V2_MONOTONIC_HORIZON",
    "selected_configuration": "C05",
    "model_sha256": actual_model_hash_10,
    "dataset_manifest_sha256": canonical_sha256_10(
        manifest_10
    ),
    "validation_npz_sha256": expected_val_hash_10,
    "split": "VALIDATION",
    "windows": int(len(Y_VAL_10)),
    "episodes": int(len(unique_episodes_10)),
    "horizons": HORIZONS_10,
    "fixed_thresholds": FIXED_THRESHOLDS_10,
    "overall_metrics": overall_rows_10,
    "bootstrap_ci_95": BOOTSTRAP_CI_10,
    "bootstrap_method": "EPISODE_CLUSTER_PERCENTILE",
    "bootstrap_replicates_requested": BOOTSTRAP_REPLICATES_10,
    "bootstrap_replicates_successful": successful_bootstraps_10,
    "horizon_monotonicity": {
        "H5_greater_than_H10_violations": (
            monotonic_violations_5_10
        ),
        "H10_greater_than_H15_violations": (
            monotonic_violations_10_15
        )
    },
    "xgboost_reference": XGBOOST_REFERENCE_10,
    "validation_training_performed": False,
    "validation_hyperparameter_optimization": False,
    "validation_recalibration_performed": False,
    "official_test_accessed": False,
    "frozen_dataset_modified": False,
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "evaluated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "limitations": [
        "Evaluation uses a controlled synthetic dataset.",
        "Overlapping windows within episodes are dependent.",
        "Bootstrap is clustered by episode rather than calibration group.",
        "XGBoost metrics are previously recorded reference values.",
        "Wholly benign episode alerts are not equivalent to all false positives.",
        "No operational threshold was optimized during this validation."
    ]
}

# Directory creation fails if anything appeared since the initial check.
REPORT_DIR_10.mkdir(
    parents=True,
    exist_ok=False
)

OVERALL_DF_10.to_csv(
    REPORT_DIR_10 / "OFFICIAL_VALIDATION_METRICS.csv",
    index=False
)

SOURCE_DF_10.to_csv(
    REPORT_DIR_10 / "SOURCE_SPECIFIC_METRICS.csv",
    index=False
)

THRESHOLD_DF_10.to_csv(
    REPORT_DIR_10 / "FIXED_THRESHOLD_METRICS.csv",
    index=False
)

CALIBRATION_DF_10.to_csv(
    REPORT_DIR_10 / "CALIBRATION_10_BINS.csv",
    index=False
)

BENIGN_DF_10.to_csv(
    REPORT_DIR_10 / "BENIGN_FALSE_ALERT_STATISTICS.csv",
    index=False
)

COMPARISON_DF_10.to_csv(
    REPORT_DIR_10 / "XGBOOST_COMPARISON.csv",
    index=False
)

# CSV is portable and does not require optional parquet libraries.
PREDICTIONS_DF_10.to_csv(
    REPORT_DIR_10 / "VALIDATION_PREDICTIONS.csv",
    index=False,
    float_format="%.17g"
)

save_json_10(
    REPORT_DIR_10
    / "EPISODE_BOOTSTRAP_CONFIDENCE_INTERVALS.json",
    BOOTSTRAP_CI_10
)

save_json_10(
    REPORT_DIR_10
    / "OFFICIAL_VALIDATION_REPORT.json",
    report_10
)

# Hash completed report files for auditability.
artifact_hashes_10 = {}

for path in sorted(REPORT_DIR_10.iterdir()):

    if path.is_file():
        artifact_hashes_10[path.name] = sha256_file_10(
            path
        )

save_json_10(
    REPORT_DIR_10 / "VALIDATION_ARTIFACT_SHA256SUMS.json",
    artifact_hashes_10
)

# ==========================================================================================
# 10Q — FINAL CONSOLE REPORT
# ==========================================================================================

print("\n" + "=" * 100)
print("LSTM V2 — OFFICIAL VALIDATION COMPLETE")
print("=" * 100)

print("\nOVERALL METRICS")

print(
    OVERALL_DF_10.round(6).to_string(
        index=False
    )
)

print("\nSOURCE-SPECIFIC METRICS")

print(
    SOURCE_DF_10.round(6).to_string(
        index=False
    )
)

print("\nFIXED-THRESHOLD METRICS")

print(
    THRESHOLD_DF_10[
        [
            "Horizon",
            "Threshold",
            "Precision",
            "Recall",
            "FP",
            "FN"
        ]
    ].round(6).to_string(
        index=False
    )
)

print("\nBENIGN FALSE ALERTS")

print(
    BENIGN_DF_10[
        [
            "Horizon",
            "Threshold",
            "Benign_Type",
            "Episodes_With_Any_Alert",
            "False_Alert_Windows",
            "Fraction_Episodes_Alerted"
        ]
    ].round(6).to_string(
        index=False
    )
)

print("\nEPISODE-BOOTSTRAP 95% CONFIDENCE INTERVALS")

for horizon in HORIZONS_10:

    h = f"H{horizon}"
    ci = BOOTSTRAP_CI_10[h]

    print(
        f"{h} AP: "
        f"[{ci['AP']['lower_95']:.6f}, "
        f"{ci['AP']['upper_95']:.6f}] | "
        f"ROC-AUC: "
        f"[{ci['ROC_AUC']['lower_95']:.6f}, "
        f"{ci['ROC_AUC']['upper_95']:.6f}]"
    )

print("\nHORIZON PROBABILITY CONSISTENCY")

print(
    "P(H5) > P(H10) violations:",
    monotonic_violations_5_10
)

print(
    "P(H10) > P(H15) violations:",
    monotonic_violations_10_15
)

print("\nXGBOOST V2 COMPARISON")

print(
    COMPARISON_DF_10.round(6).to_string(
        index=False
    )
)

print("\nReports saved to:")
print(REPORT_DIR_10)

print("\nFinal LSTM weights: UNCHANGED")
print("Additional training: NONE")
print("Hyperparameter optimization: NONE")
print("Official TEST: NOT ACCESSED")
print("Frozen dataset: UNCHANGED")

print("\n" + "=" * 100)
print("OFFICIAL LSTM VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 100)
